# Repurchase Prediction Pipeline

**Business Goal:** Predict which customers are likely to churn after their current purchase, enabling proactive retention campaigns.

## Key Question
**Given a customer just made a purchase, will they purchase again within 12 weeks?**

## Why This Matters
- **Customer retention is 5-25x cheaper than acquisition** - focusing on existing customers maximizes ROI
- **Early intervention prevents churn** - predictive models enable proactive retention before customers leave
- **Segment-specific strategies** - Golden Whales need different treatment than Casual Walk-ins
- **Data-driven prioritization** - focus limited marketing budgets on highest-value at-risk customers

## Key points of this approach
- **Order-level modeling** - Each purchase is a prediction opportunity (not just one prediction per user)
- **Temporal features with no data leakage** - Features calculated using only historical data available at prediction time
- **Multi-level analysis** - Order-level predictions → User-level insights → Segment-level strategies
- **End-to-end pipeline** - From raw data to ROI calculations and business recommendations

## Pipeline Overview
1. **Feature Engineering** - Create 30+ temporal features per order (RFM, velocity, engagement)
2. **Model Training** - Train and compare 4 algorithms (Logistic Regression, Random Forest, XGBoost, LightGBM)
3. **Model Evaluation** - Select best model using PR-AUC (optimized for imbalanced data)
4. **Performance Visualizations** - ROC curves, precision-recall, confusion matrices, segment analysis
5. **SHAP Analysis (Order-Level)** - Understand which features drive predictions for each transaction
6. **SHAP Analysis (User-Level)** - Aggregate to customer-level insights for business action
7. **Revenue Impact** - Calculate expected revenue at risk by segment and propensity level
8. **Value Estimation** - Quantify ROI for 3 scenarios: AOV uplift, churn reduction, subscription conversion

## Configuration

Set key parameters that control the pipeline behavior.

In [91]:
# Configuration flags
SKIP_FEATURE_ENGINEERING = False  # Set to True to reuse existing features (faster iteration)
PREDICTION_WEEKS = 12  # Prediction window: Will user repurchase within N weeks? (12 weeks ≈ 3 months)
FORCE_MODEL = None  # Override model selection: 'LogisticRegression', 'RandomForest', 'XGBoost', 'LightGBM', or None (auto-select best)

# Paths
DATA_PATH = "../original_data"
CLUSTER_PATH = "../cluster01/sg_user.csv"
OUTPUT_DIR = "outputs/repurchase"
MODELS_PATH = "outputs/repurchase/models"
PLOTS_PATH = "outputs/repurchase/visualizations"

# Model configuration
RANDOM_STATE = 42
TARGET_NAME = 'will_repurchase'

# Segment names from Case 1 clustering
SEGMENT_NAMES = {
    0: 'Casual Walk-in',
    1: 'Golden Whales',
    2: 'High Potential',
    3: 'Drifting Risk'
}

In [92]:
import os
import sys
import json
import logging
from pathlib import Path
from datetime import datetime, timedelta

import pandas as pd
import numpy as np
from tqdm import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)

# Tree-based models
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# SHAP
import shap
import joblib

# Setup
sns.set_style("darkgrid")
sns.set_palette("husl")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
plt.set_loglevel('warning')

# Setup logging
os.makedirs(OUTPUT_DIR, exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler(f'{OUTPUT_DIR}/pipeline.log', mode='w')
    ]
)
logger = logging.getLogger(__name__)

logger.info("Setup completed successfully")

2026-02-12 23:06:42,059 - INFO - Setup completed successfully


## Step 1: Feature Engineering

Create order-level features predicting if a user will repurchase within 12 weeks.

**Approach:**
- **One entry per order** (not per user) - each row represents a purchase moment
- **Temporal features** calculated using only data BEFORE each order (prevents data leakage)
- **Target variable**: Will this user purchase again within the prediction window?

**Key Feature Categories:**
1. **Recency features** - Days since last purchase, days since first purchase
2. **Frequency features** - Total orders, purchase velocity (orders per week)
3. **Monetary features** - Average order value, total lifetime value, spending trends
4. **Behavioral features** - Time between orders, order value changes, shipping preferences
5. **Temporal features** - Day of week, time of day, seasonality patterns

In [93]:
def load_and_clean_orders(data_path=DATA_PATH, singapore_only=True):
    """Load and clean orders dataset."""
    logger.info("Loading and cleaning orders data...")

    orders = pd.read_csv(f"{data_path}/orders.csv")

    # Filter to completed orders only
    orders = orders[orders["status"].isin(["Shipped", "Complete"])]
    orders = orders[orders["total_incl_tax"] > 0]

    # Fix datetime columns
    orders["date_placed"] = pd.to_datetime(orders["date_placed"], errors="coerce", utc=True).dt.tz_localize(None)
    orders["shipping_date"] = pd.to_datetime(orders["shipping_date"], errors="coerce", utc=True).dt.tz_localize(None)

    # Drop rows with missing critical data
    orders = orders.dropna(subset=["date_placed", "user_id"])

    logger.info(f" Cleaned orders: {len(orders):,} rows")

    # Filter to Singapore orders only
    if singapore_only:
        orders_before = len(orders)
        orders = orders[orders['currency'] == 'SGD']
        logger.info(f" Filtered to Singapore (SGD): {len(orders):,} rows (removed {orders_before - len(orders):,})")

    logger.info(f"  Date range: {orders['date_placed'].min().date()} to {orders['date_placed'].max().date()}")
    logger.info(f"  Unique users: {orders['user_id'].nunique():,}")

    return orders

In [94]:
def load_all_datasets(data_path=DATA_PATH):
    """Load all required datasets."""
    logger.info("Loading all datasets...")

    datasets = {}

    # Load main datasets
    datasets['orders'] = load_and_clean_orders(data_path)
    datasets['users'] = pd.read_csv(f"{data_path}/users.csv")
    datasets['subscriptions'] = pd.read_csv(f"{data_path}/subscriptions.csv")
    datasets['products'] = pd.read_csv(f"{data_path}/products.csv")
    datasets['events'] = pd.read_csv(f"{data_path}/events.csv")
    datasets['voucher_applications'] = pd.read_csv(f"{data_path}/voucher_applications.csv")

    # Clean datetime columns
    if 'date_signed_up' in datasets['users'].columns:
        datasets['users']['date_signed_up'] = pd.to_datetime(
            datasets['users']['date_signed_up'], errors="coerce"
        )

    # Clean events
    datasets['events']['timestamp'] = pd.to_datetime(
        datasets['events']['timestamp'], errors="coerce", utc=True
    ).dt.tz_localize(None)
    datasets['events'] = datasets['events'].dropna(subset=['user_id', 'timestamp'])

    # Clean subscriptions
    for col in ['last_order', 'next_order', 'created', 'updated']:
        if col in datasets['subscriptions'].columns:
            datasets['subscriptions'][col] = pd.to_datetime(
                datasets['subscriptions'][col], errors="coerce", utc=True
            ).dt.tz_localize(None)

    # Clean voucher applications
    datasets['voucher_applications']['created'] = pd.to_datetime(
        datasets['voucher_applications']['created'], errors="coerce", utc=True
    ).dt.tz_localize(None)

    logger.info(f" All datasets loaded")

    return datasets

In [95]:
def get_week_start(purchase_date):
    """Get the Monday of the week for a given date."""
    days_since_monday = purchase_date.weekday()
    week_start = purchase_date - timedelta(days=days_since_monday)
    return week_start

In [96]:
def create_order_based_entries(orders_df, prediction_weeks=PREDICTION_WEEKS):
    """
    Create one entry per order with target: Will user repurchase within prediction_weeks?
    """
    logger.info("="*80)
    logger.info("CREATING ORDER-BASED ENTRIES")
    logger.info("="*80)

    # Get unique orders
    orders_unique = orders_df.groupby(['user_id', 'id', 'date_placed']).agg({
        'total_incl_tax': 'first',
        'shipping_method': 'first'
    }).reset_index()

    # Add week start
    orders_unique['week_start'] = orders_unique['date_placed'].apply(get_week_start)

    logger.info(f"Total orders: {len(orders_unique):,}")
    logger.info(f"Unique users: {orders_unique['user_id'].nunique():,}")
    logger.info(f"Date range: {orders_unique['date_placed'].min().date()} to {orders_unique['date_placed'].max().date()}")

    # Create target: Will user repurchase within prediction window?
    # Logic: For each order, look forward in time to see if user makes another purchase
    logger.info(f"\nCreating {prediction_weeks}-week lookforward targets...")

    orders_sorted = orders_df[['user_id', 'date_placed']].sort_values(['user_id', 'date_placed']).copy()

    targets_dict = {}
    prediction_window = pd.Timedelta(weeks=prediction_weeks)

    for user_id, user_orders in tqdm(orders_sorted.groupby('user_id'), desc="Processing users"):
        user_order_dates = user_orders['date_placed'].values

        for i in range(len(user_order_dates)):
            order_date = user_order_dates[i]

            # Check if there's a NEXT order within the prediction window
            # Example: If order is on Jan 1 and prediction_weeks=12, check if next order is before Mar 26
            if i + 1 < len(user_order_dates):
                next_order_date = user_order_dates[i + 1]
                prediction_end = order_date + prediction_window
                has_repurchase = next_order_date <= prediction_end
            else:
                # Last order for this user = no future repurchase (label = 0)
                has_repurchase = False

            targets_dict[(user_id, pd.Timestamp(order_date))] = 1 if has_repurchase else 0

    # Map targets back
    orders_unique['will_repurchase'] = orders_unique.apply(
        lambda row: targets_dict.get((row['user_id'], row['date_placed']), 0),
        axis=1
    )

    # Calculate statistics
    n_positive = orders_unique['will_repurchase'].sum()
    n_total = len(orders_unique)
    positive_rate = n_positive / n_total

    logger.info(f"\n Targets created:")
    logger.info(f"  Total orders: {n_total:,}")
    logger.info(f"  Will repurchase: {n_positive:,} ({positive_rate:.1%})")
    logger.info(f"  Won't repurchase: {n_total - n_positive:,} ({1-positive_rate:.1%})")

    return orders_unique

In [97]:
def calculate_single_order_features(order_row, hist_orders, hist_events,
                                     hist_subs, user_info, hist_vouchers, order_date):
    """
    Calculate features for a single order using ONLY historical data (no data leakage).

    Critical: All features must be calculated using data BEFORE this order's date.
    This ensures the model only uses information that would be available at prediction time.
    Example: If order is on Jan 15, only use data from Jan 14 and earlier.
    """
    features = {
        'user_id': order_row['user_id'],
        'order_id': order_row['id'],
        'order_date': order_row['date_placed'],
        'week_start': order_row['week_start'],
        'current_order_value': order_row['total_incl_tax'],
        'current_shipping_method': order_row['shipping_method'],
        'will_repurchase': order_row['will_repurchase']
    }

    # Purchase history features
    if len(hist_orders) == 0:
        features['is_first_order'] = 1
        features['previous_orders_count'] = 0
        features['days_since_last_order'] = 9999
        features['avg_order_value_historical'] = 0
        features['total_revenue_historical'] = 0
        features['customer_tenure_days'] = 0
        features['avg_days_between_orders'] = 0
        for days in [30, 60, 90]:
            features[f'orders_last_{days}d'] = 0
            features[f'spend_last_{days}d'] = 0
    else:
        features['is_first_order'] = 0
        features['previous_orders_count'] = len(hist_orders)
        features['days_since_last_order'] = (order_date - hist_orders['date_placed'].max()).days
        features['avg_order_value_historical'] = hist_orders['total_incl_tax'].mean()
        features['total_revenue_historical'] = hist_orders['total_incl_tax'].sum()
        features['customer_tenure_days'] = (order_date - hist_orders['date_placed'].min()).days

        # Purchase velocity: Average gap between orders (in days)
        # Formula: Total tenure / (number of orders + 1) to account for current order
        # Lower value = higher frequency customer
        features['avg_days_between_orders'] = features['customer_tenure_days'] / (len(hist_orders) + 1)

        # Rolling window features: Recent activity indicators
        # Captures short-term engagement trends (30/60/90 days)
        for days in [30, 60, 90]:
            window_start = order_date - pd.Timedelta(days=days)
            window_orders = hist_orders[hist_orders['date_placed'] >= window_start]
            features[f'orders_last_{days}d'] = len(window_orders)
            features[f'spend_last_{days}d'] = window_orders['total_incl_tax'].sum() if len(window_orders) > 0 else 0

    # Event features
    if len(hist_events) == 0:
        features['total_events_historical'] = 0
        for days in [30, 60, 90]:
            features[f'events_last_{days}d'] = 0
        features['high_intent_events'] = 0
    else:
        features['total_events_historical'] = len(hist_events)
        for days in [30, 60, 90]:
            window_start = order_date - pd.Timedelta(days=days)
            features[f'events_last_{days}d'] = len(hist_events[hist_events['timestamp'] >= window_start])

        # High-intent events: Actions that signal purchase interest
        # Add to cart, view product, apply voucher = strong buying signals
        high_intent = ['ACTION_ADD_ITEM_TO_CART', 'ACTION_VIEW_PRODUCT', 'ACTION_VOUCHER_APPLY']
        features['high_intent_events'] = len(hist_events[hist_events['event'].isin(high_intent)])

    # Subscription features
    features['has_subscription'] = 1 if len(hist_subs) > 0 else 0
    features['total_subscriptions'] = len(hist_subs)

    # Voucher features
    features['total_vouchers_used'] = len(hist_vouchers)
    features['voucher_user'] = 1 if len(hist_vouchers) > 0 else 0

    # User lifecycle features
    if len(user_info) > 0:
        user_row = user_info.iloc[0]
        if pd.notna(user_row.get('date_signed_up')):
            features['customer_age_days'] = (order_date - user_row['date_signed_up']).days
        else:
            features['customer_age_days'] = 0
        features['is_singapore'] = 1 if user_row.get('country') == 'Singapore' else 0
        features['is_active_account'] = 1 if user_row.get('active') == 'Yes' else 0
    else:
        features['customer_age_days'] = 0
        features['is_singapore'] = 0
        features['is_active_account'] = 0

    return features

In [98]:
def calculate_features_at_order_time(order_entries, orders_df, events_df,
                                      subscriptions_df, users_df, voucher_df):
    """Calculate features for each order using only data BEFORE that order."""
    logger.info("="*80)
    logger.info("CALCULATING FEATURES AT ORDER TIME")
    logger.info("="*80)

    orders_sorted = orders_df.sort_values('date_placed')
    events_sorted = events_df.sort_values('timestamp')

    all_features = []

    logger.info(f"Processing {order_entries['user_id'].nunique():,} unique users...")

    for user_id, user_order_entries in tqdm(order_entries.groupby('user_id'), desc="Processing users"):
        # Get all data for this user
        user_all_orders = orders_sorted[orders_sorted['user_id'] == user_id]
        user_all_events = events_sorted[events_sorted['user_id'] == user_id]
        user_subs = subscriptions_df[subscriptions_df['user_id'] == user_id]
        user_vouchers = voucher_df[voucher_df['user_id'] == user_id]
        user_info = users_df[users_df['id'] == user_id]

        # For each order by this user
        for idx, order_row in user_order_entries.iterrows():
            order_date = order_row['date_placed']

            # Get historical data BEFORE this order
            hist_orders = user_all_orders[user_all_orders['date_placed'] < order_date]
            hist_events = user_all_events[user_all_events['timestamp'] < order_date]
            hist_subs = user_subs[user_subs['created'] < order_date]
            hist_vouchers = user_vouchers[user_vouchers['created'] < order_date]

            # Calculate features
            features = calculate_single_order_features(
                order_row, hist_orders, hist_events, hist_subs,
                user_info, hist_vouchers, order_date
            )

            all_features.append(features)

    features_df = pd.DataFrame(all_features)

    logger.info(f"\n Features calculated: {len(features_df):,} orders × {len(features_df.columns)} features")

    return features_df

In [99]:
def quantile_train_test_split(df, test_quantile=0.8):
    """Split by week_start quantile (no data leakage)."""
    logger.info("="*80)
    logger.info("QUANTILE-BASED TRAIN/TEST SPLIT")
    logger.info("="*80)

    cutoff_date = df['week_start'].quantile(test_quantile)

    train_df = df[df['week_start'] <= cutoff_date].copy()
    test_df = df[df['week_start'] > cutoff_date].copy()

    logger.info(f"\nCutoff date: {cutoff_date.date()}")
    logger.info(f"Training set: {len(train_df):,} orders ({len(train_df)/len(df):.1%})")
    logger.info(f"  Positive rate: {train_df['will_repurchase'].mean():.1%}")
    logger.info(f"Test set: {len(test_df):,} orders ({len(test_df)/len(df):.1%})")
    logger.info(f"  Positive rate: {test_df['will_repurchase'].mean():.1%}")

    return train_df, test_df

In [100]:
def run_repurchase_feature_engineering(prediction_weeks=12, output_dir=OUTPUT_DIR):
    """Run complete order-based feature engineering pipeline."""
    logger.info("="*80)
    logger.info("REPURCHASE FEATURE ENGINEERING")
    logger.info("="*80)

    os.makedirs(output_dir, exist_ok=True)

    # Load data
    datasets = load_all_datasets()

    # Create order-based entries
    order_entries = create_order_based_entries(datasets['orders'], prediction_weeks)

    # Calculate features
    features_df = calculate_features_at_order_time(
        order_entries, datasets['orders'], datasets['events'],
        datasets['subscriptions'], datasets['users'], datasets['voucher_applications']
    )

    # Handle missing values
    numeric_cols = features_df.select_dtypes(include=[np.number]).columns
    features_df[numeric_cols] = features_df[numeric_cols].fillna(0)
    features_df = features_df.replace([np.inf, -np.inf], 0)

    # Train/test split
    train_df, test_df = quantile_train_test_split(features_df, test_quantile=0.80)

    # Save features
    output_path = f"{output_dir}/repurchase_features.csv"
    features_df.to_csv(output_path, index=False)
    train_df.to_csv(f"{output_dir}/repurchase_train.csv", index=False)
    test_df.to_csv(f"{output_dir}/repurchase_test.csv", index=False)

    logger.info(f"\n Saved features to: {output_path}")
    logger.info(f" Features: {len(features_df.columns) - 7}")

    return features_df, train_df, test_df

In [101]:
# Run feature engineering if needed
if not SKIP_FEATURE_ENGINEERING:
    features_df, train_df, test_df = run_repurchase_feature_engineering(
        prediction_weeks=PREDICTION_WEEKS,
        output_dir=OUTPUT_DIR
    )
    display(features_df.head())
else:
    logger.info("Skipping feature engineering (using existing features)")

2026-02-12 23:06:42,117 - INFO - ================================================================================
2026-02-12 23:06:42,117 - INFO - REPURCHASE FEATURE ENGINEERING
2026-02-12 23:06:42,118 - INFO - ================================================================================
2026-02-12 23:06:42,118 - INFO - Loading all datasets...
2026-02-12 23:06:42,118 - INFO - Loading and cleaning orders data...
2026-02-12 23:06:42,645 - INFO -  Cleaned orders: 119,864 rows
2026-02-12 23:06:42,655 - INFO -  Filtered to Singapore (SGD): 119,085 rows (removed 779)
2026-02-12 23:06:42,656 - INFO -   Date range: 2020-01-01 to 2023-12-31
2026-02-12 23:06:42,657 - INFO -   Unique users: 30,699


/var/folders/p9/y1sq2fh51qqgly35_4sgdq180000gn/T/ipykernel_39760/1510440690.py:30: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  datasets['subscriptions'][col] = pd.to_datetime(
/var/folders/p9/y1sq2fh51qqgly35_4sgdq180000gn/T/ipykernel_39760/1510440690.py:30: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  datasets['subscriptions'][col] = pd.to_datetime(
/var/folders/p9/y1sq2fh51qqgly35_4sgdq180000gn/T/ipykernel_39760/1510440690.py:30: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  datasets['subscriptions'][col] = pd.to_datetime(


2026-02-12 23:06:43,803 - INFO -  All datasets loaded
2026-02-12 23:06:43,804 - INFO - ================================================================================
2026-02-12 23:06:43,805 - INFO - CREATING ORDER-BASED ENTRIES
2026-02-12 23:06:43,805 - INFO - ================================================================================
2026-02-12 23:06:44,031 - INFO - Total orders: 119,085
2026-02-12 23:06:44,033 - INFO - Unique users: 30,699
2026-02-12 23:06:44,033 - INFO - Date range: 2020-01-01 to 2023-12-31
2026-02-12 23:06:44,034 - INFO - 
Creating 12-week lookforward targets...


Processing users: 100%|██████████| 30699/30699 [00:00<00:00, 42469.39it/s]


2026-02-12 23:06:45,084 - INFO - 
 Targets created:
2026-02-12 23:06:45,084 - INFO -   Total orders: 119,085
2026-02-12 23:06:45,084 - INFO -   Will repurchase: 77,731 (65.3%)
2026-02-12 23:06:45,085 - INFO -   Won't repurchase: 41,354 (34.7%)
2026-02-12 23:06:45,088 - INFO - ================================================================================
2026-02-12 23:06:45,088 - INFO - CALCULATING FEATURES AT ORDER TIME
2026-02-12 23:06:45,089 - INFO - ================================================================================
2026-02-12 23:06:45,126 - INFO - Processing 30,699 unique users...


Processing users: 100%|██████████| 30699/30699 [02:13<00:00, 229.12it/s]


2026-02-12 23:08:59,418 - INFO - 
 Features calculated: 119,085 orders × 32 features
2026-02-12 23:08:59,453 - INFO - ================================================================================
2026-02-12 23:08:59,453 - INFO - QUANTILE-BASED TRAIN/TEST SPLIT
2026-02-12 23:08:59,453 - INFO - ================================================================================
2026-02-12 23:08:59,459 - INFO - 
Cutoff date: 2022-11-28
2026-02-12 23:08:59,460 - INFO - Training set: 95,268 orders (80.0%)
2026-02-12 23:08:59,460 - INFO -   Positive rate: 66.7%
2026-02-12 23:08:59,460 - INFO - Test set: 23,817 orders (20.0%)
2026-02-12 23:08:59,460 - INFO -   Positive rate: 59.8%
2026-02-12 23:09:00,333 - INFO - 
 Saved features to: outputs/repurchase/repurchase_features.csv
2026-02-12 23:09:00,333 - INFO -  Features: 25


,user_id,order_id,order_date,week_start,current_order_value,current_shipping_method,will_repurchase,is_first_order,previous_orders_count,days_since_last_order,avg_order_value_historical,total_revenue_historical,customer_tenure_days,avg_days_between_orders,orders_last_30d,spend_last_30d,orders_last_60d,spend_last_60d,orders_last_90d,spend_last_90d,total_events_historical,events_last_30d,events_last_60d,events_last_90d,high_intent_events,has_subscription,total_subscriptions,total_vouchers_used,voucher_user,customer_age_days,is_singapore,is_active_account
0,86499.0,133930,2022-05-27 05:01:38.122712,2022-05-23 05:01:38.122712,2.0,singpost,0,1,0,9999,0.0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0,0,0,0,0,0,0,0,0,1,0
1,86500.0,131756,2022-04-20 05:47:56.214103,2022-04-18 05:47:56.214103,2.0,singpost,1,1,0,9999,0.0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0,0,0,0,0,0,0,0,0,1,0
2,86500.0,131820,2022-04-21 02:01:09.731424,2022-04-18 02:01:09.731424,14.0,singpost,0,0,1,0,2.0,2.0,0,0.0,1,2.0,1,2.0,1,2.0,0,0,0,0,0,0,0,0,0,1,1,0
3,86501.0,136165,2022-05-05 05:10:24.281083,2022-05-02 05:10:24.281083,2.0,singpost,0,1,0,9999,0.0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,2,0,0,0,0,0,0,0,0,0,1,0
4,86502.0,132527,2022-04-25 10:38:11.602767,2022-04-25 10:38:11.602767,2.0,singpost,0,1,0,9999,0.0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0,0,0,0,0,0,0,0,0,1,0


## Step 2: Model Training

Train and compare multiple classification algorithms to predict repurchase behavior.

**Models evaluated:**
1. **Logistic Regression** - Linear model (scaled features), interpretable baseline
2. **Random Forest** - Tree ensemble, handles non-linear relationships
3. **XGBoost** - Gradient boosting, typically strong performance
4. **LightGBM** - Fast gradient boosting, efficient for large datasets

**Evaluation metrics:**
- **ROC-AUC**: Overall discrimination ability (threshold-independent)
- **PR-AUC**: Precision-recall tradeoff (important for imbalanced classes)
- **F1 Score**: Balance of precision and recall at default threshold

**Model selection**: Choose model with highest ROC-AUC (or force specific model via FORCE_MODEL)

In [102]:
def load_train_test_data():
    """Load training and test data."""
    train_df = pd.read_csv(f"{OUTPUT_DIR}/repurchase_train.csv")
    test_df = pd.read_csv(f"{OUTPUT_DIR}/repurchase_test.csv")

    logger.info(f"Training set: {len(train_df):,} orders")
    logger.info(f"  Positive rate: {train_df['will_repurchase'].mean():.1%}")
    logger.info(f"Test set: {len(test_df):,} orders")
    logger.info(f"  Positive rate: {test_df['will_repurchase'].mean():.1%}")

    return train_df, test_df

In [103]:
def prepare_features(df, exclude_cols=['user_id', 'order_id', 'order_date', 'week_start',
                                        'current_shipping_method', 'will_repurchase']):
    """Prepare features for modeling."""
    y = df['will_repurchase'].values

    feature_cols = [col for col in df.columns if col not in exclude_cols]
    X = df[feature_cols].copy()

    X = X.fillna(0)
    X = X.replace([np.inf, -np.inf], 0)

    feature_names = X.columns.tolist()

    return X.values, y, feature_names

In [104]:
def get_model_configs(n_pos, n_neg):
    """Get model configurations."""
    scale_pos_weight = n_neg / n_pos if n_pos > 0 else 1

    models = {
        'LogisticRegression': {
            'model': LogisticRegression(
                max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced',
                solver='lbfgs', n_jobs=1
            ),
            'requires_scaling': True
        },
        'RandomForest': {
            'model': RandomForestClassifier(
                n_estimators=100, max_depth=15, min_samples_split=5,
                random_state=RANDOM_STATE, class_weight='balanced', n_jobs=1
            ),
            'requires_scaling': False
        },
        'XGBoost': {
            'model': XGBClassifier(
                n_estimators=100, max_depth=6, learning_rate=0.1,
                random_state=RANDOM_STATE, scale_pos_weight=scale_pos_weight,
                eval_metric='logloss', n_jobs=1
            ),
            'requires_scaling': False
        },
        'LightGBM': {
            'model': LGBMClassifier(
                n_estimators=100, max_depth=6, learning_rate=0.1,
                random_state=RANDOM_STATE, class_weight='balanced',
                n_jobs=1, verbose=-1
            ),
            'requires_scaling': False
        }
    }

    return models

In [105]:
def train_all_models(X_train, y_train):
    """Train all models."""
    logger.info("="*80)
    logger.info("MODEL TRAINING")
    logger.info("="*80)

    n_pos = y_train.sum()
    n_neg = len(y_train) - n_pos

    model_configs = get_model_configs(n_pos, n_neg)
    trained_models = {}

    for model_name, config in model_configs.items():
        logger.info(f"\n  Training {model_name}...")

        model = config['model']
        requires_scaling = config['requires_scaling']

        scaler = None
        if requires_scaling:
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            model.fit(X_train_scaled, y_train)
        else:
            model.fit(X_train, y_train)

        logger.info(f"     {model_name} trained")

        trained_models[model_name] = {
            'model': model,
            'scaler': scaler,
            'requires_scaling': requires_scaling
        }

    logger.info("\n All models trained")

    return trained_models

In [106]:
# Load data and train models
train_df, test_df = load_train_test_data()

X_train, y_train, feature_names = prepare_features(train_df)
X_test, y_test, _ = prepare_features(test_df)

logger.info(f"Training features: {X_train.shape}")
logger.info(f"Test features: {X_test.shape}")
logger.info(f"Feature count: {len(feature_names)}")

user_ids_test = test_df['user_id']

trained_models = train_all_models(X_train, y_train)

2026-02-12 23:09:00,515 - INFO - Training set: 95,268 orders
2026-02-12 23:09:00,516 - INFO -   Positive rate: 66.7%
2026-02-12 23:09:00,516 - INFO - Test set: 23,817 orders
2026-02-12 23:09:00,516 - INFO -   Positive rate: 59.8%
2026-02-12 23:09:00,523 - INFO - Training features: (95268, 26)
2026-02-12 23:09:00,523 - INFO - Test features: (23817, 26)
2026-02-12 23:09:00,524 - INFO - Feature count: 26
2026-02-12 23:09:00,524 - INFO - ================================================================================
2026-02-12 23:09:00,524 - INFO - MODEL TRAINING
2026-02-12 23:09:00,524 - INFO - ================================================================================
2026-02-12 23:09:00,524 - INFO - 
  Training LogisticRegression...
2026-02-12 23:09:00,659 - INFO -      LogisticRegression trained
2026-02-12 23:09:00,659 - INFO - 
  Training RandomForest...


/Users/rickyvu/dev/DSIP_Final/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


2026-02-12 23:09:06,782 - INFO -      RandomForest trained
2026-02-12 23:09:06,782 - INFO - 
  Training XGBoost...
2026-02-12 23:09:07,158 - INFO -      XGBoost trained
2026-02-12 23:09:07,158 - INFO - 
  Training LightGBM...
2026-02-12 23:09:07,563 - INFO -      LightGBM trained
2026-02-12 23:09:07,564 - INFO - 
 All models trained


## Step 3: Model Evaluation & Selection

Evaluate all trained models on the test set and select the best performer.

**Evaluation metrics:**
- **Accuracy**: Overall correctness (less useful for imbalanced data)
- **Precision**: P(actually churned | predicted to churn) - minimizes wasted retention spend
- **Recall**: P(predicted to churn | actually churned) - minimizes missed opportunities
- **F1 Score**: Harmonic mean of precision and recall
- **ROC-AUC**: Threshold-independent discrimination measure
- **PR-AUC**: Precision-recall tradeoff (most important for imbalanced classes)

**Selection criteria:**
- Default: Choose model with highest **PR-AUC** (best for imbalanced repurchase data)
- Can override via FORCE_MODEL parameter if needed

**Why PR-AUC over ROC-AUC?**
- Repurchase is often imbalanced (e.g., 30% positive, 70% negative)
- ROC-AUC can be overly optimistic on imbalanced data
- PR-AUC directly shows precision-recall tradeoff relevant to business decisions

In [107]:
def evaluate_model(model, X_test, y_test, scaler=None):
    """Evaluate a single model."""
    if scaler is not None:
        X_test = scaler.transform(X_test)

    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1_score': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_pred_proba),
        'pr_auc': average_precision_score(y_test, y_pred_proba)
    }

    cm = confusion_matrix(y_test, y_pred)
    metrics['confusion_matrix'] = cm
    metrics['true_negatives'] = int(cm[0, 0])
    metrics['false_positives'] = int(cm[0, 1])
    metrics['false_negatives'] = int(cm[1, 0])
    metrics['true_positives'] = int(cm[1, 1])

    return metrics

In [108]:
def evaluate_all_models(trained_models, X_test, y_test):
    """Evaluate all models and store predictions."""
    logger.info("="*80)
    logger.info("MODEL EVALUATION")
    logger.info("="*80)

    results = {}

    for model_name, model_info in trained_models.items():
        logger.info(f"\n  Evaluating {model_name}...")

        metrics = evaluate_model(model_info['model'], X_test, y_test, model_info['scaler'])

        # Get predictions for curves
        if model_info['scaler'] is not None:
            X_test_scaled = model_info['scaler'].transform(X_test)
            y_proba = model_info['model'].predict_proba(X_test_scaled)[:, 1]
        else:
            y_proba = model_info['model'].predict_proba(X_test)[:, 1]

        metrics['y_test'] = y_test
        metrics['y_proba'] = y_proba

        results[model_name] = metrics

        logger.info(f"    Accuracy:  {metrics['accuracy']:.4f}")
        logger.info(f"    Precision: {metrics['precision']:.4f}")
        logger.info(f"    Recall:    {metrics['recall']:.4f}")
        logger.info(f"    F1 Score:  {metrics['f1_score']:.4f}")
        logger.info(f"    ROC-AUC:   {metrics['roc_auc']:.4f}")
        logger.info(f"    PR-AUC:    {metrics['pr_auc']:.4f}")

    return results

In [109]:
def create_comparison_table(results):
    """Create comparison table of all models."""
    comparison_data = []

    for model_name, metrics in results.items():
        comparison_data.append({
            'Model': model_name,
            'Accuracy': metrics['accuracy'],
            'Precision': metrics['precision'],
            'Recall': metrics['recall'],
            'F1': metrics['f1_score'],
            'ROC-AUC': metrics['roc_auc'],
            'PR-AUC': metrics['pr_auc']
        })

    comparison_df = pd.DataFrame(comparison_data)
    comparison_df = comparison_df.sort_values('PR-AUC', ascending=False)

    return comparison_df

In [110]:
def select_best_model(results, trained_models, metric='pr_auc', force_model=None):
    """Select best model or use forced model."""
    if force_model is not None:
        if force_model not in trained_models:
            raise ValueError(f"Model '{force_model}' not found. Available: {list(trained_models.keys())}")

        logger.info(f"\n USING SPECIFIED MODEL: {force_model}")
        selected_model_name = force_model
        selected_model_info = trained_models[force_model]
        selected_metrics = results[force_model]
    else:
        selected_model_name = max(results, key=lambda x: results[x][metric])
        selected_metrics = results[selected_model_name]
        selected_model_info = trained_models[selected_model_name]

        logger.info(f"\nBest Model: {selected_model_name}")
        logger.info(f"   Selection metric: {metric.upper()} = {selected_metrics[metric]:.4f}")

    logger.info(f"\n   All metrics:")
    logger.info(f"     Accuracy:  {selected_metrics['accuracy']:.4f}")
    logger.info(f"     Precision: {selected_metrics['precision']:.4f}")
    logger.info(f"     Recall:    {selected_metrics['recall']:.4f}")
    logger.info(f"     F1 Score:  {selected_metrics['f1_score']:.4f}")
    logger.info(f"     ROC-AUC:   {selected_metrics['roc_auc']:.4f}")
    logger.info(f"     PR-AUC:    {selected_metrics['pr_auc']:.4f}")

    return selected_model_name, selected_model_info, selected_metrics

In [111]:
def save_model_artifacts(best_model_name, best_model_info, best_metrics, feature_names):
    """Save model artifacts."""
    os.makedirs(MODELS_PATH, exist_ok=True)

    # Save model
    model_path = f"{MODELS_PATH}/{TARGET_NAME}_best_model.pkl"
    joblib.dump(best_model_info['model'], model_path)

    # Save scaler
    if best_model_info['scaler'] is not None:
        scaler_path = f"{MODELS_PATH}/{TARGET_NAME}_scaler.pkl"
        joblib.dump(best_model_info['scaler'], scaler_path)

    # Save metadata
    metadata = {
        'model_name': best_model_name,
        'requires_scaling': best_model_info['requires_scaling'],
        'feature_names': feature_names,
        'metrics': {
            'accuracy': float(best_metrics['accuracy']),
            'precision': float(best_metrics['precision']),
            'recall': float(best_metrics['recall']),
            'f1_score': float(best_metrics['f1_score']),
            'roc_auc': float(best_metrics['roc_auc']),
            'pr_auc': float(best_metrics['pr_auc'])
        },
        'created_at': datetime.now().isoformat()
    }

    metadata_path = f"{MODELS_PATH}/{TARGET_NAME}_metadata.json"
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)

    logger.info(f" Saved model artifacts to: {MODELS_PATH}")

In [112]:
# Evaluate models and select best
results = evaluate_all_models(trained_models, X_test, y_test)
comparison_df = create_comparison_table(results)

logger.info("\n" + "="*80)
logger.info("MODEL COMPARISON")
logger.info("="*80)
display(comparison_df)

best_model_name, best_model_info, best_metrics = select_best_model(
    results, trained_models, metric='pr_auc', force_model=FORCE_MODEL
)

# Get best model predictions
if best_model_info['scaler'] is not None:
    X_test_scaled = best_model_info['scaler'].transform(X_test)
    best_y_proba = best_model_info['model'].predict_proba(X_test_scaled)[:, 1]
else:
    best_y_proba = best_model_info['model'].predict_proba(X_test)[:, 1]

# Save artifacts
save_model_artifacts(best_model_name, best_model_info, best_metrics, feature_names)

2026-02-12 23:09:07,584 - INFO - ================================================================================
2026-02-12 23:09:07,584 - INFO - MODEL EVALUATION
2026-02-12 23:09:07,585 - INFO - ================================================================================
2026-02-12 23:09:07,585 - INFO - 
  Evaluating LogisticRegression...
2026-02-12 23:09:07,595 - INFO -     Accuracy:  0.7198
2026-02-12 23:09:07,595 - INFO -     Precision: 0.7447
2026-02-12 23:09:07,596 - INFO -     Recall:    0.8080
2026-02-12 23:09:07,596 - INFO -     F1 Score:  0.7751
2026-02-12 23:09:07,596 - INFO -     ROC-AUC:   0.7862
2026-02-12 23:09:07,596 - INFO -     PR-AUC:    0.8411
2026-02-12 23:09:07,596 - INFO - 
  Evaluating RandomForest...
2026-02-12 23:09:07,951 - INFO -     Accuracy:  0.7428
2026-02-12 23:09:07,951 - INFO -     Precision: 0.7392
2026-02-12 23:09:07,951 - INFO -     Recall:    0.8800
2026-02-12 23:09:07,952 - INFO -     F1 Score:  0.8035
2026-02-12 23:09:07,952 - INFO -     ROC

/Users/rickyvu/dev/DSIP_Final/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/rickyvu/dev/DSIP_Final/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/rickyvu/dev/DSIP_Final/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,Model,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
2,XGBoost,0.748919,0.773556,0.819760,0.795988,0.815451,0.860672
3,LightGBM,0.747365,0.772818,0.817511,0.794536,0.813875,0.859340
1,RandomForest,0.742789,0.739213,0.879980,0.803477,0.801450,0.848630
0,LogisticRegression,0.719780,0.744737,0.807954,0.775059,0.786219,0.841075


2026-02-12 23:09:08,140 - INFO - 
Best Model: XGBoost
2026-02-12 23:09:08,140 - INFO -    Selection metric: PR_AUC = 0.8607
2026-02-12 23:09:08,141 - INFO - 
   All metrics:
2026-02-12 23:09:08,141 - INFO -      Accuracy:  0.7489
2026-02-12 23:09:08,141 - INFO -      Precision: 0.7736
2026-02-12 23:09:08,141 - INFO -      Recall:    0.8198
2026-02-12 23:09:08,141 - INFO -      F1 Score:  0.7960
2026-02-12 23:09:08,141 - INFO -      ROC-AUC:   0.8155
2026-02-12 23:09:08,141 - INFO -      PR-AUC:    0.8607
2026-02-12 23:09:08,157 - INFO -  Saved model artifacts to: outputs/repurchase/models


## Step 4: Model Performance Visualizations

Create visual diagnostics to evaluate model performance and compare algorithms.

**Plots generated:**
1. **ROC Curves** - Shows true positive rate vs. false positive rate
   - Area under curve (AUC) measures discrimination ability
   - Higher AUC = better at separating churners from non-churners

2. **Precision-Recall Curves** - Important for imbalanced datasets
   - Precision: Of predicted churners, how many actually churned?
   - Recall: Of actual churners, how many did we predict?
   - PR-AUC often more informative than ROC-AUC for rare events

3. **Confusion Matrix** - Actual vs. predicted counts (best model only)
   - Shows false positives (wasted retention spend) and false negatives (missed churners)

4. **Model Comparison** - Bar charts comparing all algorithms on key metrics

5. **Segment Analysis** - Propensity distribution across customer segments
   - Helps identify which segments are most at-risk

In [113]:
def plot_roc_curves(results, target_name='will_repurchase', save_path=PLOTS_PATH):
    """Plot ROC curves for all models."""
    from sklearn.metrics import roc_curve

    logger.info("Creating ROC curves plot...")

    fig, ax = plt.subplots(figsize=(10, 8))
    colors = plt.cm.Set1(np.linspace(0, 1, len(results)))

    for (model_name, metrics), color in zip(results.items(), colors):
        if 'y_test' in metrics and 'y_proba' in metrics:
            fpr, tpr, _ = roc_curve(metrics['y_test'], metrics['y_proba'])
            auc = metrics['roc_auc']
            ax.plot(fpr, tpr, color=color, lw=2, label=f'{model_name} (AUC = {auc:.4f})')

    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC = 0.5)')
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title('ROC Curves - Model Comparison', fontsize=14, fontweight='bold')
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    os.makedirs(save_path, exist_ok=True)
    file_path = f"{save_path}/{target_name}_roc_curves.png"
    plt.savefig(file_path, dpi=300, bbox_inches='tight')
    plt.close()

    logger.info(f"   Saved ROC curves to: {file_path}")
    return file_path

def plot_pr_curves(results, target_name='will_repurchase', save_path=PLOTS_PATH):
    """Plot Precision-Recall curves for all models."""
    from sklearn.metrics import precision_recall_curve

    logger.info("Creating PR curves plot...")

    fig, ax = plt.subplots(figsize=(10, 8))
    colors = plt.cm.Set1(np.linspace(0, 1, len(results)))

    for (model_name, metrics), color in zip(results.items(), colors):
        if 'y_test' in metrics and 'y_proba' in metrics:
            precision, recall, _ = precision_recall_curve(metrics['y_test'], metrics['y_proba'])
            ap = metrics['pr_auc']
            ax.plot(recall, precision, color=color, lw=2, label=f'{model_name} (AP = {ap:.4f})')

    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('Recall', fontsize=12)
    ax.set_ylabel('Precision', fontsize=12)
    ax.set_title('Precision-Recall Curves - Model Comparison', fontsize=14, fontweight='bold')
    ax.legend(loc='lower left', fontsize=10)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    file_path = f"{save_path}/{target_name}_pr_curves.png"
    plt.savefig(file_path, dpi=300, bbox_inches='tight')
    plt.close()

    logger.info(f"   Saved PR curves to: {file_path}")
    return file_path

def plot_confusion_matrix_heatmap(y_test, y_pred, model_name, target_name='will_repurchase', save_path=PLOTS_PATH):
    """Plot confusion matrix heatmap."""
    cm = confusion_matrix(y_test, y_pred)

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Won\'t Repurchase', 'Will Repurchase'],
                yticklabels=['Won\'t Repurchase', 'Will Repurchase'])
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('Actual', fontsize=12)
    ax.set_title(f'Confusion Matrix - {model_name}', fontsize=14, fontweight='bold')

    plt.tight_layout()
    file_path = f"{save_path}/{target_name}_confusion_matrix.png"
    plt.savefig(file_path, dpi=300, bbox_inches='tight')
    plt.close()

    logger.info(f"   Saved confusion matrix to: {file_path}")
    return file_path

def plot_model_comparison_bars(comparison_df, target_name='will_repurchase', save_path=PLOTS_PATH):
    """Plot bar chart comparing all models."""
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    fig.suptitle('Model Performance Comparison', fontsize=16, fontweight='bold')

    metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC', 'PR-AUC']

    for idx, metric in enumerate(metrics):
        row = idx // 3
        col = idx % 3
        ax = axes[row, col]

        comparison_df.plot(x='Model', y=metric, kind='bar', ax=ax, legend=False, color='steelblue')
        ax.set_title(metric, fontsize=12, fontweight='bold')
        ax.set_xlabel('')
        ax.set_ylabel(metric, fontsize=10)
        ax.tick_params(axis='x', rotation=45)
        ax.grid(True, alpha=0.3, axis='y')

        # Add value labels
        for container in ax.containers:
            ax.bar_label(container, fmt='%.3f', fontsize=8)

    plt.tight_layout()
    file_path = f"{save_path}/{target_name}_model_comparison.png"
    plt.savefig(file_path, dpi=300, bbox_inches='tight')
    plt.close()

    logger.info(f"   Saved model comparison to: {file_path}")
    return file_path

In [114]:
# Generate evaluation plots
os.makedirs(PLOTS_PATH, exist_ok=True)

logger.info("\n" + "="*80)
logger.info("GENERATING EVALUATION PLOTS")
logger.info("="*80)

plot_roc_curves(results, TARGET_NAME, PLOTS_PATH)
plot_pr_curves(results, TARGET_NAME, PLOTS_PATH)

# Best model confusion matrix
if best_model_info['scaler'] is not None:
    X_test_scaled = best_model_info['scaler'].transform(X_test)
    best_y_pred = best_model_info['model'].predict(X_test_scaled)
else:
    best_y_pred = best_model_info['model'].predict(X_test)

plot_confusion_matrix_heatmap(y_test, best_y_pred, best_model_name, TARGET_NAME, PLOTS_PATH)
plot_model_comparison_bars(comparison_df, TARGET_NAME, PLOTS_PATH)

logger.info(" All evaluation plots generated")

2026-02-12 23:09:08,168 - INFO - 
2026-02-12 23:09:08,168 - INFO - GENERATING EVALUATION PLOTS
2026-02-12 23:09:08,168 - INFO - ================================================================================
2026-02-12 23:09:08,168 - INFO - Creating ROC curves plot...
2026-02-12 23:09:08,319 - INFO -    Saved ROC curves to: outputs/repurchase/visualizations/will_repurchase_roc_curves.png
2026-02-12 23:09:08,319 - INFO - Creating PR curves plot...
2026-02-12 23:09:08,449 - INFO -    Saved PR curves to: outputs/repurchase/visualizations/will_repurchase_pr_curves.png
2026-02-12 23:09:08,530 - INFO -    Saved confusion matrix to: outputs/repurchase/visualizations/will_repurchase_confusion_matrix.png
2026-02-12 23:09:08,853 - INFO -    Saved model comparison to: outputs/repurchase/visualizations/will_repurchase_model_comparison.png
2026-02-12 23:09:08,854 - INFO -  All evaluation plots generated


## Step 5: SHAP Analysis (Order-Level)

Use SHAP (SHapley Additive exPlanations) to understand model predictions.

**What is SHAP?**
- SHAP values explain each prediction by showing how much each feature contributed
- Based on game theory (Shapley values) - fair attribution of prediction to features
- Positive SHAP = feature pushes prediction toward "will repurchase"
- Negative SHAP = feature pushes prediction toward "won't repurchase"

**Analysis levels:**
1. **Order-level** (this section): SHAP for each order in test set
2. **User-level** (next section): Aggregate to one SHAP per customer (using most recent order)
3. **Segment-level**: Compare feature importance across customer segments

**Outputs:**
- Summary plots showing top features by importance
- Bar charts with mean |SHAP| values
- Per-segment analysis to identify segment-specific drivers

In [115]:
def calculate_shap_values(model, X_test, model_name):
    """Calculate SHAP values for model interpretability."""
    logger.info("="*80)
    logger.info("CALCULATING SHAP VALUES")
    logger.info("="*80)

    logger.info(f"  Model: {model_name}")
    logger.info(f"  Test samples: {len(X_test)}")

    # Select appropriate explainer
    if 'Logistic' in model_name:
        logger.info("  Using LinearExplainer...")
        explainer = shap.LinearExplainer(model, X_test)
        shap_values = explainer.shap_values(X_test)
    elif 'Random Forest' in model_name or 'XGBoost' in model_name or 'LightGBM' in model_name:
        logger.info("  Using TreeExplainer...")
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_test)
        if isinstance(shap_values, list):
            shap_values = shap_values[1]
    else:
        logger.info("  Using KernelExplainer...")
        background = shap.sample(X_test, min(100, len(X_test)))
        explainer = shap.KernelExplainer(model.predict_proba, background)
        shap_values = explainer.shap_values(X_test)
        if isinstance(shap_values, list):
            shap_values = shap_values[1]

    logger.info(f"   SHAP values calculated: shape {shap_values.shape}")

    return shap_values, explainer

def plot_shap_summary(shap_values, X_test, feature_names, target_name='will_repurchase', save_path=PLOTS_PATH):
    """Plot SHAP summary plot."""
    logger.info("Creating SHAP summary plot...")

    fig, ax = plt.subplots(figsize=(12, 8))
    shap.summary_plot(shap_values, X_test, feature_names=feature_names, show=False, max_display=20)
    plt.title('SHAP Summary Plot - Top 20 Features', fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()

    file_path = f"{save_path}/{target_name}_shap_summary.png"
    plt.savefig(file_path, dpi=300, bbox_inches='tight')
    plt.close()

    logger.info(f"   Saved SHAP summary to: {file_path}")
    return file_path

def plot_shap_bar(shap_values, feature_names, target_name='will_repurchase', save_path=PLOTS_PATH):
    """Plot SHAP bar plot showing feature importance."""
    logger.info("Creating SHAP bar plot...")

    # Calculate mean absolute SHAP values
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    feature_importance = pd.DataFrame({
        'feature': feature_names,
        'importance': mean_abs_shap
    }).sort_values('importance', ascending=True).tail(20)

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(feature_importance['feature'], feature_importance['importance'], color='steelblue')
    ax.set_xlabel('Mean |SHAP Value|', fontsize=12)
    ax.set_title('Feature Importance (SHAP) - Top 20', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')

    plt.tight_layout()
    file_path = f"{save_path}/{target_name}_shap_bar.png"
    plt.savefig(file_path, dpi=300, bbox_inches='tight')
    plt.close()

    logger.info(f"   Saved SHAP bar plot to: {file_path}")
    return file_path

def plot_propensity_distribution(y_test, y_proba, target_name='will_repurchase', save_path=PLOTS_PATH):
    """Plot distribution of propensity scores by actual class."""
    logger.info("Creating propensity distribution plot...")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Distribution by actual class
    repurchase_probs = y_proba[y_test == 1]
    no_repurchase_probs = y_proba[y_test == 0]

    axes[0].hist(no_repurchase_probs, bins=50, alpha=0.7, label='No Repurchase', color='#e74c3c')
    axes[0].hist(repurchase_probs, bins=50, alpha=0.7, label='Repurchase', color='#2ecc71')
    axes[0].set_xlabel('Predicted Probability', fontsize=11)
    axes[0].set_ylabel('Count', fontsize=11)
    axes[0].set_title('Probability Distribution by Actual Class', fontsize=12)
    axes[0].legend(fontsize=10)
    axes[0].axvline(x=0.3, color='orange', linestyle='--', alpha=0.7)
    axes[0].axvline(x=0.6, color='green', linestyle='--', alpha=0.7)

    # Overall distribution with segments
    axes[1].hist(y_proba, bins=50, alpha=0.7, color='#3498db')
    axes[1].axvline(x=0.3, color='orange', linestyle='--', linewidth=2, label='30% threshold')
    axes[1].axvline(x=0.6, color='green', linestyle='--', linewidth=2, label='60% threshold')
    axes[1].set_xlabel('Predicted Probability', fontsize=11)
    axes[1].set_ylabel('Count', fontsize=11)
    axes[1].set_title('Overall Probability Distribution', fontsize=12)
    axes[1].legend(fontsize=10)

    # Add segment labels
    axes[1].text(0.15, axes[1].get_ylim()[1]*0.9, 'Low\nPropensity', ha='center', fontsize=10, color='#e74c3c')
    axes[1].text(0.45, axes[1].get_ylim()[1]*0.9, 'Medium\nPropensity', ha='center', fontsize=10, color='#f39c12')
    axes[1].text(0.8, axes[1].get_ylim()[1]*0.9, 'High\nPropensity', ha='center', fontsize=10, color='#2ecc71')

    plt.tight_layout()

    os.makedirs(save_path, exist_ok=True)
    file_path = f"{save_path}/{target_name}_propensity_distribution.png"
    plt.savefig(file_path, dpi=150, bbox_inches='tight')
    plt.close()

    logger.info(f"   Saved propensity distribution to: {file_path}")
    return file_path

def plot_segment_propensity_boxplot(propensity_df, target_name='will_repurchase', save_path=PLOTS_PATH):
    """Plot box plot of propensity scores by segment."""
    logger.info("Creating segment propensity box plot...")

    fig, ax = plt.subplots(figsize=(12, 6))

    propensity_df = propensity_df.copy()
    propensity_df['segment_name'] = propensity_df['value_cluster'].map(SEGMENT_NAMES)

    # Order segments - show all 4
    order = ['Casual Walk-in', 'Golden Whales', 'High Potential', 'Drifting Risk']

    # Add missing segments with NaN values
    for segment in order:
        if segment not in propensity_df['segment_name'].values:
            dummy_row = pd.DataFrame({
                'segment_name': [segment],
                'propensity_score': [np.nan],
                'value_cluster': [[k for k, v in SEGMENT_NAMES.items() if v == segment][0]]
            })
            propensity_df = pd.concat([propensity_df, dummy_row], ignore_index=True)

    # Create box plot
    sns.boxplot(data=propensity_df, x='segment_name', y='propensity_score',
                palette='Set2', ax=ax, order=order)

    ax.set_xlabel('Customer Segment (Case 1)', fontsize=12)
    ax.set_ylabel('Repurchase Propensity Score', fontsize=12)
    ax.set_title('Repurchase Propensity Distribution by Segment', fontsize=14, fontweight='bold')
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()

    os.makedirs(save_path, exist_ok=True)
    file_path = f"{save_path}/{target_name}_segment_boxplot.png"
    plt.savefig(file_path, dpi=300, bbox_inches='tight')
    plt.close()

    logger.info(f"   Saved segment box plot to: {file_path}")
    return file_path

def plot_segment_propensity_violin(propensity_df, target_name='will_repurchase', save_path=PLOTS_PATH):
    """Plot violin plot of propensity scores by segment."""
    logger.info("Creating segment propensity violin plot...")

    fig, ax = plt.subplots(figsize=(12, 6))

    propensity_df = propensity_df.copy()
    propensity_df['segment_name'] = propensity_df['value_cluster'].map(SEGMENT_NAMES)

    # Order segments - show all 4
    order = ['Casual Walk-in', 'Golden Whales', 'High Potential', 'Drifting Risk']

    # Add missing segments with NaN values
    for segment in order:
        if segment not in propensity_df['segment_name'].values:
            dummy_row = pd.DataFrame({
                'segment_name': [segment],
                'propensity_score': [np.nan],
                'value_cluster': [[k for k, v in SEGMENT_NAMES.items() if v == segment][0]]
            })
            propensity_df = pd.concat([propensity_df, dummy_row], ignore_index=True)

    # Create violin plot
    sns.violinplot(data=propensity_df, x='segment_name', y='propensity_score',
                   palette='Set2', ax=ax, inner='box', order=order)

    ax.set_xlabel('Customer Segment (Case 1)', fontsize=12)
    ax.set_ylabel('Repurchase Propensity Score', fontsize=12)
    ax.set_title('Repurchase Propensity Density by Segment', fontsize=14, fontweight='bold')
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()

    os.makedirs(save_path, exist_ok=True)
    file_path = f"{save_path}/{target_name}_segment_violin.png"
    plt.savefig(file_path, dpi=300, bbox_inches='tight')
    plt.close()

    logger.info(f"   Saved segment violin plot to: {file_path}")
    return file_path

def plot_segment_propensity_heatmap(propensity_df, target_name='will_repurchase', save_path=PLOTS_PATH):
    """Plot heatmap of segment × propensity bins."""
    logger.info("Creating segment × propensity heatmap...")

    propensity_df = propensity_df.copy()

    # Create propensity bins
    propensity_df['propensity_bin'] = pd.cut(
        propensity_df['propensity_score'],
        bins=[0, 0.3, 0.6, 1.0],
        labels=['Low (0-0.3)', 'Medium (0.3-0.6)', 'High (0.6-1.0)']
    )

    propensity_df['segment_name'] = propensity_df['value_cluster'].map(SEGMENT_NAMES)

    # Create cross-tabulation
    cross_tab = pd.crosstab(
        propensity_df['segment_name'],
        propensity_df['propensity_bin'],
        normalize='index'
    ) * 100

    # Reorder rows - show all 4 segments
    row_order = ['Casual Walk-in', 'Golden Whales', 'High Potential', 'Drifting Risk']
    cross_tab = cross_tab.reindex(row_order, fill_value=0)

    fig, ax = plt.subplots(figsize=(10, 6))

    # Create heatmap
    sns.heatmap(cross_tab, annot=True, fmt='.1f', cmap='YlOrRd',
                ax=ax, cbar_kws={'label': '% of Segment'})

    ax.set_xlabel('Propensity Level', fontsize=12)
    ax.set_ylabel('Customer Segment', fontsize=12)
    ax.set_title('Segment × Repurchase Propensity Distribution (%)', fontsize=14, fontweight='bold')

    plt.tight_layout()

    os.makedirs(save_path, exist_ok=True)
    file_path = f"{save_path}/{target_name}_segment_heatmap.png"
    plt.savefig(file_path, dpi=300, bbox_inches='tight')
    plt.close()

    logger.info(f"   Saved segment heatmap to: {file_path}")
    return file_path

In [116]:
# Generate propensity distribution and segment-specific visualizations
logger.info("\n" + "="*80)
logger.info("SEGMENT-SPECIFIC VISUALIZATIONS")
logger.info("="*80)

# Plot overall propensity distribution
plot_propensity_distribution(y_test, best_y_proba, TARGET_NAME, PLOTS_PATH)

# Create propensity DataFrame for segment analysis
logger.info("\nCreating propensity DataFrame for segment analysis...")

# Load cluster data
cluster_df = pd.read_csv(CLUSTER_PATH)
if 'user_id' not in cluster_df.columns:
    cluster_df = cluster_df.rename(columns={'id': 'user_id'})

# Create propensity df from test set
propensity_df = test_df[['user_id']].copy()
propensity_df['propensity_score'] = best_y_proba

# Merge with cluster data
propensity_df = propensity_df.merge(cluster_df[['user_id', 'value_cluster']], on='user_id', how='left')

logger.info(f"  Created propensity_df with {len(propensity_df)} records")
logger.info(f"  Matched {propensity_df['value_cluster'].notna().sum()} users with segments")

# Generate segment plots
plot_segment_propensity_boxplot(propensity_df, TARGET_NAME, PLOTS_PATH)
plot_segment_propensity_violin(propensity_df, TARGET_NAME, PLOTS_PATH)
plot_segment_propensity_heatmap(propensity_df, TARGET_NAME, PLOTS_PATH)

2026-02-12 23:09:08,868 - INFO - 
2026-02-12 23:09:08,869 - INFO - SEGMENT-SPECIFIC VISUALIZATIONS
2026-02-12 23:09:08,869 - INFO - ================================================================================
2026-02-12 23:09:08,870 - INFO - Creating propensity distribution plot...
2026-02-12 23:09:09,009 - INFO -    Saved propensity distribution to: outputs/repurchase/visualizations/will_repurchase_propensity_distribution.png
2026-02-12 23:09:09,009 - INFO - 
Creating propensity DataFrame for segment analysis...
2026-02-12 23:09:09,065 - INFO -   Created propensity_df with 23817 records
2026-02-12 23:09:09,065 - INFO -   Matched 23817 users with segments
2026-02-12 23:09:09,065 - INFO - Creating segment propensity box plot...
2026-02-12 23:09:09,191 - INFO -    Saved segment box plot to: outputs/repurchase/visualizations/will_repurchase_segment_boxplot.png
2026-02-12 23:09:09,191 - INFO - Creating segment propensity violin plot...


/var/folders/p9/y1sq2fh51qqgly35_4sgdq180000gn/T/ipykernel_39760/3599124212.py:140: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=propensity_df, x='segment_name', y='propensity_score',
/var/folders/p9/y1sq2fh51qqgly35_4sgdq180000gn/T/ipykernel_39760/3599124212.py:182: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=propensity_df, x='segment_name', y='propensity_score',


2026-02-12 23:09:09,320 - INFO -    Saved segment violin plot to: outputs/repurchase/visualizations/will_repurchase_segment_violin.png
2026-02-12 23:09:09,320 - INFO - Creating segment × propensity heatmap...
2026-02-12 23:09:09,423 - INFO -    Saved segment heatmap to: outputs/repurchase/visualizations/will_repurchase_segment_heatmap.png


'outputs/repurchase/visualizations/will_repurchase_segment_heatmap.png'

In [117]:
# Calculate and plot SHAP values
logger.info("\n" + "="*80)
logger.info("SHAP ANALYSIS")
logger.info("="*80)

# Prepare data for SHAP
if best_model_info['scaler'] is not None:
    X_test_for_shap = best_model_info['scaler'].transform(X_test)
else:
    X_test_for_shap = X_test

shap_values, explainer = calculate_shap_values(best_model_info['model'], X_test_for_shap, best_model_name)

# Generate overall SHAP plots
plot_shap_summary(shap_values, X_test_for_shap, feature_names, TARGET_NAME, PLOTS_PATH)
plot_shap_bar(shap_values, feature_names, TARGET_NAME, PLOTS_PATH)

# Generate segment-specific SHAP analysis
logger.info("\n" + "-"*80)
logger.info("SEGMENT-SPECIFIC SHAP ANALYSIS")
logger.info("-"*80)

# Create SHAP DataFrame with user info
shap_df = pd.DataFrame(shap_values, columns=feature_names)
shap_df['user_id'] = test_df['user_id'].values

# Merge with segmentation
shap_df = shap_df.merge(cluster_df[['user_id', 'value_cluster']], on='user_id', how='left')
matched = shap_df['value_cluster'].notna().sum()
logger.info(f"Matched {matched:,} / {len(shap_df):,} users with segments")

# Create mapping of user_id to segment for filtering
user_to_segment = dict(zip(cluster_df['user_id'], cluster_df['value_cluster']))
segment_labels = np.array([user_to_segment.get(uid, -1) for uid in test_df['user_id'].values])

# Analyze and plot for each segment
segment_results = {}
for segment_id, segment_name in SEGMENT_NAMES.items():
    logger.info(f"\nSegment: {segment_name} (Cluster {segment_id})")

    # Filter to segment
    segment_mask = segment_labels == segment_id
    segment_shap = shap_values[segment_mask]

    if len(segment_shap) == 0:
        logger.info(f"  No samples in this segment")
        continue

    logger.info(f"  Samples: {len(segment_shap):,}")

    # Calculate mean absolute SHAP for segment
    mean_abs_shap = np.abs(segment_shap).mean(axis=0)
    mean_shap = segment_shap.mean(axis=0)

    # Create importance DataFrame
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'mean_abs_shap': mean_abs_shap,
        'mean_shap': mean_shap
    }).sort_values('mean_abs_shap', ascending=False)
    importance_df['rank'] = range(1, len(importance_df) + 1)

    segment_results[segment_name] = {
        'segment_id': segment_id,
        'n_samples': len(segment_shap),
        'importance': importance_df,
        'shap_values': segment_shap
    }

    # Log top 5 drivers
    logger.info("  Top 5 drivers:")
    for _, row in importance_df.head(5).iterrows():
        direction = "+" if row['mean_shap'] > 0 else "-"
        logger.info(f"    {row['rank']:2d}. {row['feature']}: {row['mean_abs_shap']:.4f} ({direction})")

    # Save segment-specific importance
    safe_name = segment_name.replace(' ', '_').replace('-', '_').lower()
    segment_importance_path = f"{MODELS_PATH}/{TARGET_NAME}_importance_{safe_name}.csv"
    importance_df.to_csv(segment_importance_path, index=False)
    logger.info(f"   Saved importance to: {segment_importance_path}")

    # Create SHAP summary plot for this segment
    logger.info(f"  Creating SHAP summary plot for {segment_name}...")

    segment_X = X_test_for_shap[segment_mask]

    plt.figure(figsize=(12, 10))
    shap.summary_plot(
        segment_shap,
        segment_X,
        feature_names=feature_names,
        show=False,
        max_display=20
    )

    plt.title(f'SHAP Feature Importance - {segment_name}\n(n={len(segment_shap):,} orders)',
             fontsize=14, fontweight='bold')
    plt.tight_layout()

    # Save
    file_path = f"{PLOTS_PATH}/{TARGET_NAME}_shap_summary_{safe_name}.png"
    plt.savefig(file_path, dpi=300, bbox_inches='tight')
    plt.close()

    logger.info(f"   Saved to: {file_path}")

logger.info(f"\n Created SHAP analysis for {len(segment_results)} segments")

# Save overall SHAP values
shap_output_path = f"{MODELS_PATH}/{TARGET_NAME}_shap_values.csv"
shap_df.to_csv(shap_output_path, index=False)
logger.info(f" Saved SHAP values to: {shap_output_path}")

2026-02-12 23:09:09,429 - INFO - 
2026-02-12 23:09:09,429 - INFO - SHAP ANALYSIS
2026-02-12 23:09:09,430 - INFO - ================================================================================
2026-02-12 23:09:09,430 - INFO - ================================================================================
2026-02-12 23:09:09,430 - INFO - CALCULATING SHAP VALUES
2026-02-12 23:09:09,430 - INFO - ================================================================================
2026-02-12 23:09:09,430 - INFO -   Model: XGBoost
2026-02-12 23:09:09,431 - INFO -   Test samples: 23817
2026-02-12 23:09:09,431 - INFO -   Using TreeExplainer...
2026-02-12 23:09:14,517 - INFO -    SHAP values calculated: shape (23817, 26)
2026-02-12 23:09:14,518 - INFO - Creating SHAP summary plot...
2026-02-12 23:09:17,203 - INFO -    Saved SHAP summary to: outputs/repurchase/visualizations/will_repurchase_shap_summary.png
2026-02-12 23:09:17,203 - INFO - Creating SHAP bar plot...
2026-02-12 23:09:17,316 - INFO 

## Step 5b: User-Level SHAP Analysis

Aggregate order-level SHAP values to customer-level for actionable business insights.

**Why user-level aggregation?**
- The model predicts at order-level, but **business actions target customers, not orders**
- Marketing campaigns reach users, not individual transactions
- Revenue impact calculations need one prediction per customer (avoid double-counting)
- Segments need equal weighting per user for fair comparison

**Aggregation method:**
- For each user, select their **MOST RECENT order** in the test set
- This represents the user's current state and latest behavior
- Matches the approach used in revenue impact analysis (consistency)

**Outputs:**
- User-level SHAP summary plots (overall + per segment)
- Feature importance bar charts at customer level
- Enables fair comparison: "What drives repurchase for Golden Whales vs Casual Walk-ins?"

In [118]:
def aggregate_order_shap_to_users(shap_values, X_test, user_ids, feature_names, test_df):
    """
    Aggregate order-level SHAP values to user-level using most recent order.

    This follows the same logic as revenue impact analysis:
    - For each user, use their MOST RECENT order's SHAP values
    - This represents the user's current state

    Args:
        shap_values: SHAP values array (order-level)
        X_test: Test features array (order-level)
        user_ids: User IDs for test set (Series or array)
        feature_names: List of feature names
        test_df: Test DataFrame with order_date

    Returns:
        pd.DataFrame: User-level DataFrame with aggregated SHAP values
    """
    logger.info("Aggregating order-level SHAP to user-level (using most recent order)...")

    # Create DataFrame with SHAP values (each row = one order)
    shap_df = pd.DataFrame(shap_values, columns=feature_names)

    # Handle user_ids as Series or array
    if hasattr(user_ids, 'values'):
        shap_df['user_id'] = user_ids.values
    else:
        shap_df['user_id'] = user_ids

    # Add order_date from test_df to enable temporal sorting
    shap_df['order_date'] = pd.to_datetime(test_df['order_date'].values)

    # Aggregate: For each user, keep ONLY their most recent order's SHAP values
    # Logic: Sort by date, then groupby user_id and take .last() = most recent
    shap_df_sorted = shap_df.sort_values('order_date')
    user_shap_df = shap_df_sorted.groupby('user_id').last().reset_index()

    # Clean up: Remove order_date (only needed for sorting)
    user_shap_df = user_shap_df.drop('order_date', axis=1)

    logger.info(f"  Orders in test set: {len(shap_df):,}")
    logger.info(f"  Unique users: {len(user_shap_df):,}")
    logger.info(f"  Avg orders per user: {len(shap_df) / len(user_shap_df):.2f}")

    return user_shap_df


def plot_user_shap_summary(user_shap_values, user_features, feature_names,
                            target_name='will_repurchase', save_path=PLOTS_PATH):
    """Create SHAP summary plot at user level."""
    logger.info("\nCreating user-level SHAP summary plot...")

    n_users = len(user_shap_values)

    plt.figure(figsize=(12, 10))
    shap.summary_plot(user_shap_values, user_features, feature_names=feature_names,
                     show=False, max_display=20)

    plt.title(f'SHAP Feature Importance - User Level\n(n={n_users:,} users)',
             fontsize=14, fontweight='bold')
    plt.tight_layout()

    os.makedirs(save_path, exist_ok=True)
    file_path = f"{save_path}/{target_name}_shap_summary_user_level.png"
    plt.savefig(file_path, dpi=300, bbox_inches='tight')
    plt.close()

    logger.info(f"   Saved to: {file_path}")
    return file_path


def plot_user_shap_bar(user_shap_df, feature_names, target_name='will_repurchase', save_path=PLOTS_PATH):
    """Plot user-level SHAP bar chart (mean absolute values)."""
    logger.info("Creating user-level SHAP bar plot...")

    # Extract SHAP values
    user_shap_values = user_shap_df[feature_names].values
    n_users = len(user_shap_values)

    # Calculate mean absolute SHAP values
    mean_abs_shap = np.abs(user_shap_values).mean(axis=0)

    # Sort and get top 20
    sorted_idx = np.argsort(mean_abs_shap)[::-1][:20]
    top_features = [feature_names[i] for i in sorted_idx]
    top_values = mean_abs_shap[sorted_idx]

    plt.figure(figsize=(10, 10))
    colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(top_features)))
    plt.barh(range(len(top_features)), top_values[::-1], color=colors[::-1])
    plt.yticks(range(len(top_features)), top_features[::-1])
    plt.xlabel('Mean |SHAP Value|', fontsize=12)
    plt.title(f'Top 20 Feature Importance (SHAP) - User Level\n(n={n_users:,} users)',
             fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()

    os.makedirs(save_path, exist_ok=True)
    file_path = f"{save_path}/{target_name}_shap_bar_user_level.png"
    plt.savefig(file_path, dpi=300, bbox_inches='tight')
    plt.close()

    logger.info(f"   Saved to: {file_path}")
    return file_path


def plot_user_shap_by_segment(user_shap_df, segmentation_df, feature_names,
                                target_name='will_repurchase', save_path=PLOTS_PATH):
    """Create per-segment SHAP plots at user level."""
    logger.info("\nCreating user-level SHAP plots by segment...")

    # Merge with segmentation
    user_data = user_shap_df.merge(segmentation_df, on='user_id', how='left')

    # Count matched users
    matched = user_data['value_cluster'].notna().sum()
    logger.info(f"  Matched {matched:,} / {len(user_data):,} users with segments")

    plot_paths = []

    for segment_id, segment_name in SEGMENT_NAMES.items():
        # Filter to segment
        segment_mask = user_data['value_cluster'] == segment_id
        segment_data = user_data[segment_mask]

        if len(segment_data) == 0:
            logger.info(f"  Skipping {segment_name} (no users)")
            continue

        n_users = len(segment_data)
        logger.info(f"  Creating SHAP summary for {segment_name} (n={n_users:,} users)")

        # Extract SHAP values and features
        segment_shap_values = segment_data[feature_names].values

        # Create SHAP summary plot
        plt.figure(figsize=(12, 10))
        shap.summary_plot(
            segment_shap_values,
            segment_shap_values,  # Use same for features (shows SHAP value distributions)
            feature_names=feature_names,
            show=False,
            max_display=20
        )

        plt.title(f'SHAP Feature Importance - {segment_name}\n(n={n_users:,} users)',
                 fontsize=14, fontweight='bold')
        plt.tight_layout()

        # Save
        os.makedirs(save_path, exist_ok=True)
        safe_name = segment_name.replace(' ', '_').replace('-', '_').lower()
        file_path = f"{save_path}/{target_name}_shap_summary_user_level_{safe_name}.png"
        plt.savefig(file_path, dpi=300, bbox_inches='tight')
        plt.close()

        plot_paths.append(file_path)
        logger.info(f"     Saved to: {file_path}")

    logger.info(f"   Created {len(plot_paths)} user-level segment SHAP plots")
    return plot_paths

In [119]:
logger.info("\n" + "="*80)
logger.info("USER-LEVEL SHAP ANALYSIS")
logger.info("="*80)

# Step 1: Aggregate to user level
user_shap_df = aggregate_order_shap_to_users(
    shap_values, X_test_for_shap, test_df['user_id'], feature_names, test_df
)

# Step 2: Load segmentation
try:
    segmentation_df = cluster_df[['user_id', 'value_cluster']]  # Already loaded earlier
except NameError:
    # If cluster_df doesn't exist, load it
    segmentation_df = pd.read_csv(CLUSTER_PATH)
    if 'user_id' not in segmentation_df.columns:
        segmentation_df = segmentation_df.rename(columns={'id': 'user_id'})
    segmentation_df = segmentation_df[['user_id', 'value_cluster']]

# Step 3: Generate user-level visualizations
user_shap_values = user_shap_df[feature_names].values

# Overall user-level SHAP summary
plot_user_shap_summary(user_shap_values, user_shap_values, feature_names, TARGET_NAME, PLOTS_PATH)

# User-level SHAP bar chart
plot_user_shap_bar(user_shap_df, feature_names, TARGET_NAME, PLOTS_PATH)

# Per-segment user-level plots
segment_plot_paths = plot_user_shap_by_segment(
    user_shap_df, segmentation_df, feature_names, TARGET_NAME, PLOTS_PATH
)

# Step 4: Save results
user_shap_results = {
    'user_shap_df': user_shap_df,
    'user_shap_values': user_shap_values,
    'feature_names': feature_names,
    'n_users': len(user_shap_df)
}

logger.info(f"\n User-level SHAP analysis complete")
logger.info(f"  Orders in test set: {len(test_df):,}")
logger.info(f"  Unique users: {user_shap_results['n_users']:,}")
logger.info(f"  Avg orders per user: {len(test_df) / user_shap_results['n_users']:.1f}")
logger.info(f"  User-level plots generated: {len(segment_plot_paths) + 2}")

# Save user-level SHAP values
user_shap_output_path = f"{MODELS_PATH}/{TARGET_NAME}_user_shap_values.csv"
user_shap_results['user_shap_df'].to_csv(user_shap_output_path, index=False)
logger.info(f"   Saved user-level SHAP values to: {user_shap_output_path}")

2026-02-12 23:09:20,604 - INFO - 
2026-02-12 23:09:20,604 - INFO - USER-LEVEL SHAP ANALYSIS
2026-02-12 23:09:20,605 - INFO - ================================================================================
2026-02-12 23:09:20,605 - INFO - Aggregating order-level SHAP to user-level (using most recent order)...
2026-02-12 23:09:20,612 - INFO -   Orders in test set: 23,817
2026-02-12 23:09:20,612 - INFO -   Unique users: 8,441
2026-02-12 23:09:20,613 - INFO -   Avg orders per user: 2.82
2026-02-12 23:09:20,613 - INFO - 
Creating user-level SHAP summary plot...
2026-02-12 23:09:21,767 - INFO -    Saved to: outputs/repurchase/visualizations/will_repurchase_shap_summary_user_level.png
2026-02-12 23:09:21,767 - INFO - Creating user-level SHAP bar plot...
2026-02-12 23:09:21,896 - INFO -    Saved to: outputs/repurchase/visualizations/will_repurchase_shap_bar_user_level.png
2026-02-12 23:09:21,897 - INFO - 
Creating user-level SHAP plots by segment...
2026-02-12 23:09:21,898 - INFO -   Matched 

## Step 6: Revenue Impact Analysis

Estimate the potential revenue impact of interventions targeting at-risk customers.

**Business question:** If we prevent 20% of predicted churners from leaving, how much revenue do we save?

**Methodology:**
1. **User-level aggregation**: One prediction per customer (uses most recent order)
2. **Segment stratification**: Analyze Golden Whales, High Potential, etc. separately
3. **Revenue calculation**: Expected value = P(repurchase) × Historical AOV × Frequency
4. **Intervention scenarios**: Model revenue saved if we reduce churn by 10%, 20%, 30%

**Key metrics:**
- **At-risk users**: Customers with propensity_score < 0.5 (predicted to NOT repurchase)
- **Expected lifetime value**: Projected revenue if user continues purchasing
- **Segment comparison**: Which segments have highest revenue-at-risk?

**Outputs:**
- Revenue impact by segment and propensity level
- Expected revenue saved under different intervention scenarios
- Prioritization: Which customer segments to target first?

In [120]:
def aggregate_to_user_level(test_df, propensity_scores, cluster_path):
    """Aggregate order-level data to user-level."""
    logger.info("Aggregating to user level...")

    # Load cluster data
    cluster_df = pd.read_csv(cluster_path)
    if 'user_id' not in cluster_df.columns:
        cluster_df = cluster_df.rename(columns={'id': 'user_id'})

    # Add propensity scores
    order_df = test_df.copy()
    order_df['propensity_score'] = propensity_scores
    order_df['order_date'] = pd.to_datetime(order_df['order_date'])

    # Get most recent order per user (represents current state for propensity)
    user_latest = order_df.sort_values('order_date').groupby('user_id').last().reset_index()

    # Calculate user-level average order value (AOV) from test set orders
    user_avg_ov = order_df.groupby('user_id')['current_order_value'].mean().reset_index()
    user_avg_ov.columns = ['user_id', 'user_avg_order_value']

    user_df = user_latest[['user_id', 'propensity_score']].copy()
    user_df = user_df.merge(user_avg_ov, on='user_id', how='left')

    # Merge with cluster data to get historical AOV (from full purchase history)
    cluster_df['correct_avg_order_value'] = cluster_df['monetary_total'] / cluster_df['order_count']
    user_df = user_df.merge(
        cluster_df[['user_id', 'value_cluster', 'correct_avg_order_value']],
        on='user_id', how='left'
    )

    # Use test set AOV if available, otherwise fallback to historical AOV from clustering
    user_df['avg_order_value'] = user_df['user_avg_order_value'].fillna(
        user_df['correct_avg_order_value']
    )

    # Map cluster IDs to human-readable segment names
    user_df['segment_name'] = user_df['value_cluster'].map(SEGMENT_NAMES)

    # Create propensity risk segments for analysis
    # Low (<30%): High churn risk - priority for retention campaigns
    # Medium (30-60%): Uncertain - may benefit from nudges
    # High (>60%): Low churn risk - less urgent intervention
    user_df['propensity_segment'] = pd.cut(
        user_df['propensity_score'],
        bins=[0, 0.3, 0.6, 1.0],
        labels=['Low (<30%)', 'Medium (30-60%)', 'High (>60%)']
    )

    logger.info(f"   User-level aggregation complete: {len(user_df):,} users")

    return user_df, order_df

In [121]:
# Default intervention configuration
DEFAULT_INTERVENTION_CONFIG = {
    'Casual Walk-in': {
        'High (>60%)': {'cost': 0.15, 'lift': 0.15},
        'Medium (30-60%)': {'cost': 0.60, 'lift': 0.25},
        'Low (<30%)': {'cost': 1.50, 'lift': 0.20}
    },
    'Golden Whales': {
        'High (>60%)': {'cost': 0.20, 'lift': 0.20},
        'Medium (30-60%)': {'cost': 0.80, 'lift': 0.30},
        'Low (<30%)': {'cost': 2.00, 'lift': 0.35}
    },
    'High Potential': {
        'High (>60%)': {'cost': 0.15, 'lift': 0.15},
        'Medium (30-60%)': {'cost': 0.70, 'lift': 0.30},
        'Low (<30%)': {'cost': 1.50, 'lift': 0.25}
    },
    'Drifting Risk': {
        'High (>60%)': {'cost': 0.12, 'lift': 0.12},
        'Medium (30-60%)': {'cost': 0.50, 'lift': 0.20},
        'Low (<30%)': {'cost': 1.20, 'lift': 0.15}
    }
}

In [122]:
def calculate_revenue_impact(user_df, intervention_config=None):
    """Calculate revenue impact by segment × propensity using odds-based lift."""
    if intervention_config is None:
        intervention_config = DEFAULT_INTERVENTION_CONFIG

    logger.info("Calculating revenue impact...")

    results = []

    for segment_name in SEGMENT_NAMES.values():
        for prop_segment in ['Low (<30%)', 'Medium (30-60%)', 'High (>60%)']:
            mask = (user_df['segment_name'] == segment_name) & (user_df['propensity_segment'] == prop_segment)
            segment_data = user_df[mask]

            if len(segment_data) == 0:
                continue

            n_users = len(segment_data)
            median_order_value = segment_data['avg_order_value'].median()

            # Get intervention parameters
            config = intervention_config[segment_name][prop_segment]
            intervention_cost = n_users * config['cost']
            lift = config['lift']

            # Calculate baseline
            avg_propensity = segment_data['propensity_score'].mean()
            baseline_repurchases = n_users * avg_propensity
            baseline_revenue = baseline_repurchases * median_order_value

            # Apply odds-based lift
            if avg_propensity >= 0.9999:
                new_propensity = 0.9999
            elif avg_propensity <= 0.0001:
                odds = 0.0001 / (1 - 0.0001)
                new_odds = odds * (1 + lift)
                new_propensity = new_odds / (1 + new_odds)
            else:
                odds = avg_propensity / (1 - avg_propensity)
                new_odds = odds * (1 + lift)
                new_propensity = new_odds / (1 + new_odds)

            intervention_repurchases = n_users * new_propensity
            intervention_revenue = intervention_repurchases * median_order_value

            # Calculate incremental impact
            incremental_repurchases = intervention_repurchases - baseline_repurchases
            incremental_revenue = intervention_revenue - baseline_revenue

            net_profit = incremental_revenue - intervention_cost
            roi = (net_profit / intervention_cost * 100) if intervention_cost > 0 else 0

            results.append({
                'value_cluster': segment_name,
                'propensity_segment': prop_segment,
                'n_users': n_users,
                'avg_propensity': avg_propensity,
                'median_order_value': median_order_value,
                'intervention_cost_per_user': config['cost'],
                'intervention_cost_total': intervention_cost,
                'expected_lift_odds': lift,
                'new_propensity': new_propensity,
                'baseline_repurchases': baseline_repurchases,
                'baseline_revenue': baseline_revenue,
                'intervention_repurchases': intervention_repurchases,
                'intervention_revenue': intervention_revenue,
                'additional_repurchases': incremental_repurchases,
                'additional_revenue': incremental_revenue,
                'net_profit': net_profit,
                'roi_percent': roi
            })

    results_df = pd.DataFrame(results)
    logger.info(f"   Calculated impact for {len(results_df)} combinations")

    return results_df

In [123]:
# Run revenue impact analysis
user_df, order_df = aggregate_to_user_level(test_df, best_y_proba, CLUSTER_PATH)
revenue_results_df = calculate_revenue_impact(user_df)

# Display results
logger.info("\n" + "="*80)
logger.info("REVENUE IMPACT SUMMARY")
logger.info("="*80)

total_profit = revenue_results_df['net_profit'].sum()
total_cost = revenue_results_df['intervention_cost_total'].sum()
total_revenue = revenue_results_df['additional_revenue'].sum()
overall_roi = (total_profit / total_cost * 100) if total_cost > 0 else 0

logger.info(f"\nAll Segments:")
logger.info(f"  Total Investment: ${total_cost:,.2f}")
logger.info(f"  Expected Revenue: ${total_revenue:,.2f}")
logger.info(f"  Net Profit: ${total_profit:,.2f}")
logger.info(f"  ROI: {overall_roi:.1f}%")

# Smart targeting (profitable only)
profitable = revenue_results_df[revenue_results_df['roi_percent'] > 0]
if len(profitable) > 0:
    smart_profit = profitable['net_profit'].sum()
    smart_cost = profitable['intervention_cost_total'].sum()
    smart_roi = (smart_profit / smart_cost * 100) if smart_cost > 0 else 0

    logger.info(f"\nSmart Targeting (ROI > 0%):")
    logger.info(f"  Profitable Segments: {len(profitable)}")
    logger.info(f"  Investment: ${smart_cost:,.2f}")
    logger.info(f"  Net Profit: ${smart_profit:,.2f}")
    logger.info(f"  ROI: {smart_roi:.1f}%")

display(revenue_results_df.nlargest(10, 'net_profit'))

2026-02-12 23:09:23,468 - INFO - Aggregating to user level...
2026-02-12 23:09:23,537 - INFO -    User-level aggregation complete: 8,441 users
2026-02-12 23:09:23,539 - INFO - Calculating revenue impact...
2026-02-12 23:09:23,544 - INFO -    Calculated impact for 9 combinations
2026-02-12 23:09:23,544 - INFO - 
2026-02-12 23:09:23,544 - INFO - REVENUE IMPACT SUMMARY
2026-02-12 23:09:23,544 - INFO - ================================================================================
2026-02-12 23:09:23,545 - INFO - 
All Segments:
2026-02-12 23:09:23,545 - INFO -   Total Investment: $8,235.20
2026-02-12 23:09:23,545 - INFO -   Expected Revenue: $4,850.20
2026-02-12 23:09:23,545 - INFO -   Net Profit: $-3,385.00
2026-02-12 23:09:23,545 - INFO -   ROI: -41.1%
2026-02-12 23:09:23,546 - INFO - 
Smart Targeting (ROI > 0%):
2026-02-12 23:09:23,546 - INFO -   Profitable Segments: 5
2026-02-12 23:09:23,546 - INFO -   Investment: $1,486.80
2026-02-12 23:09:23,546 - INFO -   Net Profit: $652.45
2026-0

,value_cluster,propensity_segment,n_users,avg_propensity,median_order_value,intervention_cost_per_user,intervention_cost_total,expected_lift_odds,new_propensity,baseline_repurchases,baseline_revenue,intervention_repurchases,intervention_revenue,additional_repurchases,additional_revenue,net_profit,roi_percent
8,High Potential,High (>60%),1782,0.803237,14.000000,0.15,267.3,0.15,0.824395,1431.368652,20039.161133,1469.072021,20567.008301,37.703369,527.847168,260.547168,97.473688
7,High Potential,Medium (30-60%),1039,0.447541,14.000000,0.70,727.3,0.30,0.512935,464.995117,6509.931641,532.939880,7461.158325,67.944763,951.226685,223.926685,30.788765
1,Casual Walk-in,Medium (30-60%),654,0.425881,14.000000,0.60,392.4,0.25,0.481126,278.526184,3899.366577,314.656219,4405.187073,36.130035,505.820496,113.420496,28.904306
4,Golden Whales,Medium (30-60%),79,0.411461,20.830000,0.80,63.2,0.30,0.476127,32.505398,677.087436,37.614010,783.499825,5.108612,106.412389,43.212389,68.374034
2,Casual Walk-in,High (>60%),244,0.736743,7.500000,0.15,36.6,0.15,0.762941,179.765396,1348.240471,186.157654,1396.182404,6.392258,47.941933,11.341933,30.988887
5,Golden Whales,High (>60%),202,0.916047,14.000000,0.20,40.4,0.20,0.929047,185.041534,2590.581482,187.667404,2627.343658,2.625870,36.762177,-3.637823,-9.004514
3,Golden Whales,Low (<30%),93,0.203223,30.414286,2.00,186.0,0.35,0.256133,18.899775,574.823143,23.820393,724.480227,4.920618,149.657083,-36.342917,-19.539202
6,High Potential,Low (<30%),1816,0.157278,23.363333,1.50,2724.0,0.25,0.189159,285.616272,6672.948168,343.513580,8025.622282,57.897308,1352.674114,-1371.325886,-50.342360
0,Casual Walk-in,Low (<30%),2532,0.137918,20.000000,1.50,3798.0,0.20,0.161059,349.207581,6984.151611,407.800537,8156.010742,58.592957,1171.859131,-2626.140869,-69.145363


In [124]:
def plot_revenue_impact_analysis(results_df, save_path):
    """Create revenue impact visualizations."""
    logger.info("Creating revenue impact visualizations...")

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    # 1. Net Profit by Segment × Propensity
    pivot_profit = results_df.pivot_table(
        values='net_profit',
        index='value_cluster',
        columns='propensity_segment'
    )

    colors = ['#e74c3c', '#f39c12', '#2ecc71']
    pivot_profit.plot(kind='bar', ax=axes[0, 0], color=colors)
    axes[0, 0].set_title('Expected Net Profit by Segment & Propensity', fontsize=13, fontweight='bold')
    axes[0, 0].set_xlabel('Customer Segment')
    axes[0, 0].set_ylabel('Net Profit ($)')
    axes[0, 0].legend(title='Propensity', bbox_to_anchor=(1.05, 1))
    axes[0, 0].axhline(y=0, color='black', linestyle='--', alpha=0.5)
    axes[0, 0].tick_params(axis='x', rotation=45)
    axes[0, 0].grid(True, alpha=0.3, axis='y')

    # 2. ROI by Segment × Propensity
    pivot_roi = results_df.pivot_table(
        values='roi_percent',
        index='value_cluster',
        columns='propensity_segment'
    )

    pivot_roi.plot(kind='bar', ax=axes[0, 1], color=colors)
    axes[0, 1].set_title('Expected ROI (%) by Segment & Propensity', fontsize=13, fontweight='bold')
    axes[0, 1].set_xlabel('Customer Segment')
    axes[0, 1].set_ylabel('ROI (%)')
    axes[0, 1].legend(title='Propensity', bbox_to_anchor=(1.05, 1))
    axes[0, 1].axhline(y=0, color='black', linestyle='--', alpha=0.5)
    axes[0, 1].tick_params(axis='x', rotation=45)
    axes[0, 1].grid(True, alpha=0.3, axis='y')

    # 3. Investment vs Return scatter
    prop_colors = {'Low (<30%)': '#e74c3c', 'Medium (30-60%)': '#f39c12', 'High (>60%)': '#2ecc71'}

    for _, row in results_df.iterrows():
        axes[1, 0].scatter(
            row['intervention_cost_total'],
            row['additional_revenue'],
            s=150, alpha=0.7,
            color=prop_colors.get(row['propensity_segment'], '#3498db'),
            edgecolors='black', linewidths=0.5
        )

    max_val = max(results_df['intervention_cost_total'].max(), results_df['additional_revenue'].max())
    axes[1, 0].plot([0, max_val * 1.1], [0, max_val * 1.1], 'k--', alpha=0.3, label='Break-even')

    axes[1, 0].set_title('Investment vs Expected Return', fontsize=13, fontweight='bold')
    axes[1, 0].set_xlabel('Total Investment ($)')
    axes[1, 0].set_ylabel('Expected Additional Revenue ($)')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].legend()

    # 4. Top opportunities
    top_10 = results_df.nlargest(10, 'net_profit').copy()
    top_10['label'] = top_10['value_cluster'].str[:12] + '\n' + top_10['propensity_segment']

    bar_colors = ['#2ecc71' if x > 0 else '#e74c3c' for x in top_10['net_profit']]
    axes[1, 1].barh(range(len(top_10)), top_10['net_profit'], color=bar_colors)
    axes[1, 1].set_yticks(range(len(top_10)))
    axes[1, 1].set_yticklabels(top_10['label'], fontsize=9)
    axes[1, 1].set_xlabel('Net Profit ($)')
    axes[1, 1].set_title('Top Opportunities by Net Profit', fontsize=13, fontweight='bold')
    axes[1, 1].axvline(x=0, color='black', linestyle='-', alpha=0.5)
    axes[1, 1].grid(True, alpha=0.3, axis='x')

    plt.tight_layout()

    file_path = f"{save_path}/revenue_impact_analysis.png"
    plt.savefig(file_path, dpi=300, bbox_inches='tight')
    plt.close()

    logger.info(f"   Saved revenue impact plot to: {file_path}")
    return file_path

def plot_user_distribution(user_df, save_path):
    """Plot user distribution across segments and propensity levels."""
    logger.info("Creating user distribution visualization...")

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    # 1. Users by segment
    segment_counts = user_df['segment_name'].value_counts()
    colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']
    segment_counts.plot(kind='bar', ax=axes[0, 0], color=colors[:len(segment_counts)])
    axes[0, 0].set_title('Users by Customer Segment', fontsize=13, fontweight='bold')
    axes[0, 0].set_xlabel('Segment')
    axes[0, 0].set_ylabel('Number of Users')
    axes[0, 0].tick_params(axis='x', rotation=45)

    for i, v in enumerate(segment_counts):
        axes[0, 0].text(i, v + 50, f'{v:,}', ha='center', fontsize=10)

    # 2. Users by propensity
    prop_counts = user_df['propensity_segment'].value_counts()
    prop_colors = {'Low (<30%)': '#e74c3c', 'Medium (30-60%)': '#f39c12', 'High (>60%)': '#2ecc71'}
    prop_counts.plot(kind='bar', ax=axes[0, 1],
                    color=[prop_colors.get(x, '#3498db') for x in prop_counts.index])
    axes[0, 1].set_title('Users by Propensity Level', fontsize=13, fontweight='bold')
    axes[0, 1].set_xlabel('Propensity Segment')
    axes[0, 1].set_ylabel('Number of Users')
    axes[0, 1].tick_params(axis='x', rotation=45)

    for i, v in enumerate(prop_counts):
        axes[0, 1].text(i, v + 50, f'{v:,}', ha='center', fontsize=10)

    # 3. Heatmap
    pivot_users = pd.crosstab(user_df['segment_name'], user_df['propensity_segment'])
    col_order = ['Low (<30%)', 'Medium (30-60%)', 'High (>60%)']
    pivot_users = pivot_users.reindex(columns=[c for c in col_order if c in pivot_users.columns])

    sns.heatmap(pivot_users, annot=True, fmt='d', cmap='YlOrRd', ax=axes[1, 0],
                cbar_kws={'label': 'Number of Users'})
    axes[1, 0].set_title('User Count by Segment × Propensity', fontsize=13, fontweight='bold')
    axes[1, 0].set_xlabel('Propensity Segment')
    axes[1, 0].set_ylabel('Customer Segment')

    # 4. Stacked bar
    pivot_users_pct = pivot_users.div(pivot_users.sum(axis=1), axis=0) * 100
    pivot_users_pct.plot(kind='barh', stacked=True, ax=axes[1, 1],
                         color=['#e74c3c', '#f39c12', '#2ecc71'])
    axes[1, 1].set_title('Propensity Distribution within Each Segment', fontsize=13, fontweight='bold')
    axes[1, 1].set_xlabel('Percentage of Users')
    axes[1, 1].set_ylabel('Customer Segment')
    axes[1, 1].legend(title='Propensity', bbox_to_anchor=(1.05, 1))

    plt.tight_layout()

    file_path = f"{save_path}/user_distribution.png"
    plt.savefig(file_path, dpi=300, bbox_inches='tight')
    plt.close()

    logger.info(f"   Saved user distribution to: {file_path}")
    return file_path

In [125]:
# Generate revenue impact visualizations
revenue_output_path = f"{OUTPUT_DIR}/revenue_impact"
os.makedirs(revenue_output_path, exist_ok=True)

plot_revenue_impact_analysis(revenue_results_df, revenue_output_path)
plot_user_distribution(user_df, revenue_output_path)

# Save results
revenue_results_df.to_csv(f"{revenue_output_path}/revenue_impact_detailed.csv", index=False)
user_df.to_csv(f"{revenue_output_path}/user_level_data.csv", index=False)
logger.info(f"\n Saved revenue impact results to: {revenue_output_path}")

2026-02-12 23:09:23,561 - INFO - Creating revenue impact visualizations...
2026-02-12 23:09:23,885 - INFO -    Saved revenue impact plot to: outputs/repurchase/revenue_impact/revenue_impact_analysis.png
2026-02-12 23:09:23,886 - INFO - Creating user distribution visualization...
2026-02-12 23:09:24,182 - INFO -    Saved user distribution to: outputs/repurchase/revenue_impact/user_distribution.png
2026-02-12 23:09:24,196 - INFO - 
 Saved revenue impact results to: outputs/repurchase/revenue_impact


## Step 7: Value Estimation Scenarios

Quantify revenue potential and ROI across 3 strategic business levers.

**Scenario 1: AOV (Average Order Value) Uplift**
- **Goal**: Increase spending per transaction via upselling, bundling, or premium offerings
- **Example**: Free shipping thresholds, product recommendations, volume discounts
- **Calculation**: (Uplift % × AOV × Purchase Frequency × Users) - Campaign Cost

**Scenario 2: Churn Reduction vs. Customer Acquisition**
- **Goal**: Compare ROI of retaining existing customers vs. acquiring new ones
- **Key insight**: Retention is often 5-25x cheaper than acquisition
- **Calculation**: Revenue saved from prevented churn vs. cost to acquire equivalent users

**Scenario 3: Subscription Model Conversion**
- **Goal**: Convert one-time buyers into recurring subscribers
- **Example**: Coffee subscription, meal kit plans, membership programs
- **Calculation**: (Subscribers × Monthly Fee × Retention Months) - Conversion Cost

**Each scenario includes:**
- Revenue impact per segment
- Cost per user (intervention/campaign costs)
- Net profit and ROI
- Visualization comparing scenarios

In [126]:
# Value estimation configuration
AOV_UPLIFT_CONFIG = {
    'Golden Whales':  {'uplift_pct': 0.075, 'cost_per_user': 3.00},
    'High Potential': {'uplift_pct': 0.15, 'cost_per_user': 1.00},
    'Casual Walk-in': {'uplift_pct': 0.10, 'cost_per_user': 0.50},
}

TARGETED_AOV_CONFIG = {
    'current_aov_range': (10, 15),     # identify users currently buying in this range
    'target_aov': 25,                  # target order value to nudge them toward
    'conversion_rate': 0.25,           # % of target users who actually increase AOV
    'campaign_cost_per_user': 0.80,    # cost to target each user in range (discount + email)
}

CHURN_REDUCTION_CONFIG = {
    'freq_uplift': {
        'Low (<30%)':      0.08,
        'Medium (30-60%)': 0.10,
        'High (>60%)':     0.05,
    },
    'retention_cost_per': {
        'Low (<30%)':      2.00,
        'Medium (30-60%)': 0.80,
        'High (>60%)':     0.30,
    },
    'acq_cost_per_user': 8.00,
    'new_customer_pct':  0.10,
}

SUBSCRIPTION_CONFIG = {
    'conversion_rate':          0.30,
    'conversion_effectiveness': 0.70,
    'campaign_cost_per_user':   1.00,
    'target_min_propensity':    0.50,
}

In [127]:
def load_baseline_data_for_value_estimation():
    """Load baseline data for value estimation scenarios."""
    features = pd.read_csv(f"{OUTPUT_DIR}/repurchase_features.csv")
    features['order_date'] = pd.to_datetime(features['order_date'])
    features['week_start'] = pd.to_datetime(features['week_start'])

    # Replicate test split
    cutoff = features['week_start'].quantile(0.8)
    test = features[features['week_start'] > cutoff].copy()

    # Test period span
    test_start = test['order_date'].min()
    test_end = test['order_date'].max()
    test_months = (test_end - test_start).days / 30.44

    # Per-user aggregation
    user_agg = test.groupby('user_id').agg(
        n_orders=('order_id', 'count'),
        aov=('current_order_value', 'mean'),
        has_subscription=('has_subscription', 'max'),
    ).reset_index()

    user_agg['freq_3mo'] = user_agg['n_orders'] * 3.0 / test_months

    # Merge with user-level data
    user_level = user_df.copy()
    user_baseline = user_agg.merge(
        user_level[['user_id', 'segment_name', 'propensity_segment', 'propensity_score']],
        on='user_id', how='inner'
    )

    logger.info(f"Loaded {len(user_baseline):,} users for value estimation")

    return user_baseline

In [128]:
def scenario_aov_uplift(user_df, config=None):
    """Scenario 1: AOV Uplift by segment."""
    if config is None:
        config = AOV_UPLIFT_CONFIG

    logger.info("\n" + "="*70)
    logger.info(" SCENARIO 1: AOV UPLIFT")
    logger.info("="*70)

    rows = []
    for seg, params in config.items():
        grp = user_df[user_df['segment_name'] == seg]
        if grp.empty:
            continue

        n = len(grp)
        freq = grp['freq_3mo'].mean()
        aov = grp['aov'].mean()
        baseline = n * freq * aov
        uplift_pct = params['uplift_pct']
        uplifted = baseline * (1 + uplift_pct)
        incremental = uplifted - baseline

        cost_per = params['cost_per_user']
        cost = n * cost_per if cost_per is not None else None
        roi = (incremental - cost) / cost * 100 if cost else None

        rows.append({
            'Segment': seg,
            'n': n,
            'freq_3mo': round(freq, 3),
            'aov': round(aov, 2),
            'baseline_revenue': baseline,
            'uplift_pct': uplift_pct,
            'uplifted_revenue': uplifted,
            'incremental_revenue': incremental,
            'cost': cost,
            'roi': roi,
        })

    df = pd.DataFrame(rows)

    logger.info(f"\n{'Segment':<18} {'n':>6} {'Freq':>7} {'AOV':>8} {'Baseline':>12} {'Uplift':>7} {'Incr. Rev':>12} {'ROI':>8}")
    logger.info("-" * 90)
    for r in rows:
        roi_s = f"{r['roi']:>6.1f}%" if r['roi'] is not None else "  N/A"
        logger.info(f"{r['Segment']:<18} {r['n']:>6,} {r['freq_3mo']:>7.3f} ${r['aov']:>7.2f} "
                   f"${r['baseline_revenue']:>10,.2f} {r['uplift_pct']*100:>5.0f}% "
                   f"${r['incremental_revenue']:>10,.2f} {roi_s}")

    return df

In [129]:
def scenario_targeted_aov_uplift(user_df, config=None):
    """
    Scenario 1b: Identify users in a specific AOV range and model revenue
    from nudging a fraction of them to a higher target AOV.
    """
    if config is None:
        config = TARGETED_AOV_CONFIG

    logger.info("\n" + "="*70)
    logger.info("SCENARIO 1b: TARGETED AOV THRESHOLD UPLIFT")
    logger.info("="*70)

    aov_min, aov_max = config['current_aov_range']
    target_aov = config['target_aov']
    conv_rate = config['conversion_rate']
    cost_per = config['campaign_cost_per_user']

    # Filter users in target AOV range
    target_group = user_df[(user_df['aov'] >= aov_min) & (user_df['aov'] < aov_max)]

    if target_group.empty:
        logger.info(f"  No users found in AOV range ${aov_min}–${aov_max}")
        return None

    n = len(target_group)
    freq = target_group['freq_3mo'].mean()
    current_aov = target_group['aov'].mean()

    # Baseline revenue
    baseline = n * freq * current_aov

    # Uplifted scenario
    n_converted = n * conv_rate
    uplifted_revenue = (n_converted * freq * target_aov) + ((n - n_converted) * freq * current_aov)
    incremental_revenue = uplifted_revenue - baseline

    # Cost and ROI
    cost = n * cost_per if cost_per is not None else None
    roi = (incremental_revenue - cost) / cost * 100 if cost else None

    logger.info(f"\nTarget group:      users with AOV ${aov_min}–${aov_max}")
    logger.info(f"                   n = {n:,}   freq = {freq:.3f}   current AOV = ${current_aov:.2f}")
    logger.info(f"Campaign:          nudge to ${target_aov} with discount/incentive")
    logger.info(f"  Conversion rate: {conv_rate*100:.0f}%")
    logger.info(f"  Users converted: {n_converted:,.0f}")
    logger.info(f"Incremental rev:   ${incremental_revenue:,.2f}")
    logger.info(f"Campaign cost:     {'N/A' if cost is None else f'${cost:,.2f}'}")
    logger.info(f"ROI:               {'N/A' if roi is None else f'{roi:.1f}%'}")

    return {
        'n': n,
        'freq': freq,
        'current_aov': current_aov,
        'target_aov': target_aov,
        'conv_rate': conv_rate,
        'n_converted': n_converted,
        'baseline': baseline,
        'uplifted_revenue': uplifted_revenue,
        'incremental_revenue': incremental_revenue,
        'cost': cost,
        'roi': roi,
    }

def plot_targeted_aov_uplift(result_data, output_path):
    """Bar chart: Incremental Revenue | Campaign Cost | Profit with ROI annotation."""
    if result_data is None:
        return

    incr = result_data['incremental_revenue']
    cost = result_data['cost']
    roi = result_data['roi']

    if cost is None:
        return

    profit = incr - cost

    labels = ['Incremental\nRevenue', 'Campaign\nCost', 'Profit']
    values = [incr, cost, profit]
    colors = ['#2ecc71', '#e74c3c', '#3498db']

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(labels, values, color=colors, width=0.5)

    ax.set_title('Scenario 1b: Targeted AOV Threshold Uplift', fontsize=14, fontweight='bold')
    ax.set_ylabel('Amount per 3 months ($)')
    ax.grid(True, axis='y', alpha=0.3)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v:,.0f}'))

    max_val = max(values) if values else 0
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                val + (max_val * 0.03 if max_val > 0 else 0.5),
                f'${val:,.0f}', ha='center', va='bottom', fontsize=10)

    # ROI annotation on profit bar
    if roi is not None:
        profit_bar = bars[-1]
        ax.text(profit_bar.get_x() + profit_bar.get_width() / 2,
                profit_bar.get_height() + (max_val * 0.12 if max_val > 0 else 1.0),
                f'ROI: {roi:.1f}%', ha='center', va='bottom',
                fontsize=11, color='#f39c12', fontweight='bold')

    plt.tight_layout()
    file_path = f"{output_path}/scenario1b_targeted_aov.png"
    plt.savefig(file_path, dpi=150, bbox_inches='tight')
    plt.close()
    logger.info(f"   Saved: {file_path}")
    return file_path

In [130]:
def scenario_churn_reduction(user_df, config=None):
    """Scenario 2: Churn Reduction vs Acquisition."""
    if config is None:
        config = CHURN_REDUCTION_CONFIG

    logger.info("\n" + "="*70)
    logger.info(" SCENARIO 2: CHURN REDUCTION vs ACQUISITION")
    logger.info("="*70)

    tiers = ['Low (<30%)', 'Medium (30-60%)', 'High (>60%)']

    tier_data = []
    for tier in tiers:
        grp = user_df[user_df['propensity_segment'] == tier]
        if grp.empty:
            continue
        tier_data.append({
            'tier': tier,
            'n': len(grp),
            'freq': grp['freq_3mo'].mean(),
            'aov': grp['aov'].mean(),
        })
        tier_data[-1]['baseline_revenue'] = tier_data[-1]['n'] * tier_data[-1]['freq'] * tier_data[-1]['aov']

    # Retention
    retention_rows = []
    for t in tier_data:
        uplift = config['freq_uplift'].get(t['tier'])
        cost_p = config['retention_cost_per'].get(t['tier'])

        if uplift is not None:
            new_freq = t['freq'] * (1 + uplift)
            ret_rev = t['n'] * new_freq * t['aov']
            ret_incr = ret_rev - t['baseline_revenue']
            ret_cost = t['n'] * cost_p if cost_p is not None else None
            ret_roi = (ret_incr - ret_cost) / ret_cost * 100 if ret_cost else None
        else:
            new_freq = ret_rev = ret_incr = ret_cost = ret_roi = None

        retention_rows.append({**t, 'uplift': uplift, 'new_freq': new_freq,
                              'ret_incr': ret_incr, 'ret_cost': ret_cost, 'ret_roi': ret_roi})

    # Acquisition
    total_n = sum(t['n'] for t in tier_data)
    freq_port = sum(t['n'] * t['freq'] for t in tier_data) / total_n
    aov_port = sum(t['n'] * t['aov'] for t in tier_data) / total_n

    new_pct = config.get('new_customer_pct')
    acq_cost_p = config.get('acq_cost_per_user')

    if new_pct is not None:
        n_new = total_n * new_pct
        acq_rev = n_new * freq_port * aov_port
        acq_cost = n_new * acq_cost_p if acq_cost_p is not None else None
        acq_roi = (acq_rev - acq_cost) / acq_cost * 100 if acq_cost else None
    else:
        n_new = acq_rev = acq_cost = acq_roi = None

    # Summary
    total_ret_incr = sum(r['ret_incr'] for r in retention_rows if r['ret_incr'] is not None)
    total_ret_cost = sum(r['ret_cost'] for r in retention_rows if r['ret_incr'] is not None and r['ret_cost'] is not None)
    total_ret_roi = (total_ret_incr - total_ret_cost) / total_ret_cost * 100 if total_ret_cost else None

    logger.info(f"\nRetention (all tiers): Revenue=${total_ret_incr:,.2f}, Cost=${total_ret_cost:,.2f}, ROI={total_ret_roi:.1f}%")
    logger.info(f"Acquisition: Revenue=${acq_rev:,.2f}, Cost=${acq_cost:,.2f}, ROI={acq_roi:.1f}%")

    return {
        'tier_data': tier_data,
        'retention_rows': retention_rows,
        'acquisition': {'n_new': n_new, 'acq_rev': acq_rev, 'acq_cost': acq_cost, 'acq_roi': acq_roi},
        'totals': {'ret_incr': total_ret_incr, 'ret_cost': total_ret_cost, 'ret_roi': total_ret_roi}
    }

In [131]:
def scenario_subscription_conversion(user_df, config=None):
    """Scenario 3: Subscription Conversion."""
    if config is None:
        config = SUBSCRIPTION_CONFIG

    logger.info("\n" + "="*70)
    logger.info(" SCENARIO 3: SUBSCRIPTION CONVERSION")
    logger.info("="*70)

    min_prop = config['target_min_propensity']

    subs = user_df[user_df['has_subscription'] == 1]
    nonsubs = user_df[user_df['has_subscription'] == 0]
    target = nonsubs[nonsubs['propensity_score'] >= min_prop]

    freq_sub = subs['freq_3mo'].mean()
    freq_target = target['freq_3mo'].mean()
    aov_target = target['aov'].mean()
    freq_gap = freq_sub - freq_target

    conv_rate = config['conversion_rate']
    eff = config['conversion_effectiveness']
    cost_p = config['campaign_cost_per_user']
    n_target = len(target)

    logger.info(f"\nSubscribers: n={len(subs):,}, freq={freq_sub:.3f}")
    logger.info(f"Target non-subs: n={n_target:,}, freq={freq_target:.3f}, aov=${aov_target:.2f}")
    logger.info(f"Frequency gap: {freq_gap:.3f}")

    if conv_rate is not None:
        n_converted = n_target * conv_rate
        freq_uplift_conv = freq_gap * eff
        freq_projected = freq_target + freq_uplift_conv
        incremental_revenue = n_converted * freq_uplift_conv * aov_target
        cost = n_target * cost_p if cost_p is not None else None
        roi = (incremental_revenue - cost) / cost * 100 if cost else None

        logger.info(f"\nConversion rate: {conv_rate*100:.0f}%")
        logger.info(f"Users converted: {n_converted:,.0f}")
        logger.info(f"Projected freq: {freq_projected:.3f}")
        logger.info(f"Incremental revenue: ${incremental_revenue:,.2f}")
        logger.info(f"ROI: {roi:.1f}%")
    else:
        n_converted = freq_uplift_conv = freq_projected = incremental_revenue = cost = roi = None

    return {
        'freq_sub': freq_sub, 'freq_target': freq_target, 'freq_gap': freq_gap,
        'aov_target': aov_target, 'n_target': n_target,
        'n_converted': n_converted, 'freq_projected': freq_projected,
        'incremental_revenue': incremental_revenue, 'roi': roi
    }

In [132]:
def plot_aov_distribution(user_df, config=None, output_path=None):
    """Plot histogram of user count by AOV bucket with segment breakdown."""
    if config is None:
        config = {
            'bucket_resolution': 5,
            'max_display_aov': 100,
            'segment_breakdown': True
        }
    if output_path is None:
        output_path = f"{OUTPUT_DIR}/value_estimation"

    resolution = config['bucket_resolution']
    max_aov = config['max_display_aov']
    segment_breakdown = config['segment_breakdown']

    logger.info("\n" + "="*70)
    logger.info("AOV DISTRIBUTION ANALYSIS")
    logger.info("="*70)
    logger.info(f"  Bucket resolution: ${resolution}")
    logger.info(f"  Display cap: ${max_aov}+")

    # Cap AOV values at max_display_aov
    user_df_plot = user_df.copy()
    user_df_plot['aov_capped'] = user_df_plot['aov'].clip(upper=max_aov)

    if segment_breakdown:
        # 3-panel figure: one subplot per segment
        segments = ['Golden Whales', 'High Potential', 'Casual Walk-in']
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))

        for i, seg in enumerate(segments):
            seg_data = user_df_plot[user_df_plot['segment_name'] == seg]
            if seg_data.empty:
                axes[i].text(0.5, 0.5, f'No data\nfor {seg}',
                            ha='center', va='center', transform=axes[i].transAxes)
                axes[i].set_title(seg, fontsize=11, fontweight='bold')
                continue

            bins = np.arange(0, max_aov + resolution, resolution)
            counts, edges = np.histogram(seg_data['aov_capped'], bins=bins)

            axes[i].bar(edges[:-1], counts, width=resolution, align='edge',
                       color='#3498db', edgecolor='white', linewidth=0.5)

            # Find and annotate peak bucket
            peak_idx = np.argmax(counts)
            peak_count = counts[peak_idx]
            peak_bucket = f"${edges[peak_idx]:.0f}–${edges[peak_idx+1]:.0f}"
            axes[i].text(edges[peak_idx] + resolution/2, peak_count + max(counts)*0.02,
                        f'{peak_count:,}', ha='center', va='bottom',
                        fontsize=8, fontweight='bold', color='#2c3e50')

            axes[i].set_title(seg, fontsize=11, fontweight='bold')
            axes[i].set_xlabel('AOV ($)', fontsize=9)
            axes[i].set_ylabel('Number of Users', fontsize=9)
            axes[i].grid(True, axis='y', alpha=0.3)
            axes[i].set_xlim(0, max_aov)

            tick_stride = max(1, int(20 / resolution))
            tick_positions = edges[::tick_stride]
            axes[i].set_xticks(tick_positions)
            axes[i].set_xticklabels([f'{int(t)}' for t in tick_positions], fontsize=8)

            logger.info(f"  {seg}: n={len(seg_data):,}, median AOV=${seg_data['aov'].median():.2f}, "
                       f"peak bucket {peak_bucket} ({peak_count:,} users)")

        fig.suptitle('AOV Distribution by Segment', fontsize=14, fontweight='bold', y=1.02)
        plt.tight_layout()
        file_path = f"{output_path}/aov_distribution.png"
        plt.savefig(file_path, dpi=150, bbox_inches='tight')
        plt.close()
        logger.info(f"   Saved: {file_path}")
        return file_path

def plot_scenario_1_profit_roi(s1_df, output_path):
    """Bar chart: incremental revenue, campaign cost and profit + ROI by segment."""
    if s1_df is None or s1_df.empty:
        return

    df = s1_df.copy()
    df = df[df['cost'].notna()]
    if df.empty:
        return

    df['profit'] = df['incremental_revenue'] - df['cost']

    segments = df['Segment'].tolist()
    incr = df['incremental_revenue'].tolist()
    cost = df['cost'].tolist()
    profit = df['profit'].tolist()
    roi_vals = df['roi'].astype(float).values

    x = np.arange(len(segments))
    width = 0.25

    fig, ax1 = plt.subplots(figsize=(10, 5))
    ax1.bar(x - width, incr, width, label='Increased Revenue vs Baseline', color='#2ecc71')
    ax1.bar(x, cost, width, label='Campaign Cost', color='#e74c3c')
    ax1.bar(x + width, profit, width, label='Incremental Profit (Rev − Cost)', color='#3498db')

    ax1.set_title('Scenario 1: Profit & ROI by Segment', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Incremental uplift / Cost / Profit per 3 months ($)')
    ax1.set_xticks(x)
    ax1.set_xticklabels(segments)
    ax1.grid(True, axis='y', alpha=0.3)
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v:,.0f}'))

    # Secondary axis for ROI
    if not np.isnan(roi_vals).all():
        ax2 = ax1.twinx()
        roi_mask = ~np.isnan(roi_vals)
        x_roi = x[roi_mask]
        roi_plot = roi_vals[roi_mask]
        ax2.plot(x_roi, roi_plot, color='#f39c12', marker='o', linestyle='-', label='ROI (%)')
        ax2.set_ylabel('ROI (%)')
        max_roi = np.nanmax(roi_vals)
        for xi, r in zip(x_roi, roi_plot):
            ax2.text(xi, r + (max_roi * 0.02 if max_roi > 0 else 0.5),
                     f'{r:.0f}%', ha='center', va='bottom', fontsize=9, color='#f39c12')

        handles1, labels1 = ax1.get_legend_handles_labels()
        handles2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(handles1 + handles2, labels1 + labels2, loc='upper left')
    else:
        ax1.legend(loc='upper left')

    plt.tight_layout()
    file_path = f"{output_path}/scenario1_profit_roi.png"
    plt.savefig(file_path, dpi=150, bbox_inches='tight')
    plt.close()
    logger.info(f"   Saved: {file_path}")
    return file_path

def plot_scenario_2_profit_roi(s2_data, output_path):
    """Two panels: per-tier and strategy-level profit & ROI."""
    tier_data = s2_data['tier_data']
    retention_rows = s2_data['retention_rows']
    totals = s2_data['totals']
    acq = s2_data['acquisition']

    rows = [r for r in retention_rows if r['ret_incr'] is not None and r['ret_cost'] is not None]
    if not rows and (totals['ret_incr'] is None and acq['acq_rev'] is None):
        return

    labels_tier = [r['tier'] for r in rows]
    incr_tier = [r['ret_incr'] for r in rows]
    cost_tier = [r['ret_cost'] for r in rows]
    profit_tier = [inc - c for inc, c in zip(incr_tier, cost_tier)]
    roi_tier = [r['ret_roi'] for r in rows]

    # Strategy-level aggregates
    strategies = []
    incr_s = []
    cost_s = []
    profit_s = []
    roi_s = []

    if totals['ret_incr'] is not None and totals['ret_cost'] is not None:
        strategies.append('Retention\n(All tiers)')
        incr_s.append(totals['ret_incr'])
        cost_s.append(totals['ret_cost'])
        profit_s.append(totals['ret_incr'] - totals['ret_cost'])
        roi_s.append(totals['ret_roi'])

    profitable = [r for r in retention_rows if r['ret_roi'] is not None and r['ret_roi'] > 0]
    if profitable:
        smart_incr = sum(r['ret_incr'] for r in profitable)
        smart_cost = sum(r['ret_cost'] for r in profitable)
        smart_profit = smart_incr - smart_cost
        smart_roi = (smart_incr - smart_cost) / smart_cost * 100 if smart_cost else None

        strategies.append('Retention\n(Med+High)')
        incr_s.append(smart_incr)
        cost_s.append(smart_cost)
        profit_s.append(smart_profit)
        roi_s.append(smart_roi)

    if acq['acq_rev'] is not None and acq['acq_cost'] is not None:
        acq_profit = acq['acq_rev'] - acq['acq_cost']
        strategies.append('Acquisition')
        incr_s.append(acq['acq_rev'])
        cost_s.append(acq['acq_cost'])
        profit_s.append(acq_profit)
        roi_s.append(acq['acq_roi'])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: per-tier
    if rows:
        x = np.arange(len(labels_tier))
        width = 0.25
        axes[0].bar(x - width, incr_tier, width, label='Incremental Revenue', color='#2ecc71')
        axes[0].bar(x, cost_tier, width, label='Campaign Cost', color='#e74c3c')
        axes[0].bar(x + width, profit_tier, width, label='Profit', color='#3498db')

        axes[0].set_title('Retention by Tier: Profit & ROI', fontsize=12, fontweight='bold')
        axes[0].set_ylabel('Amount per 3 months ($)')
        axes[0].set_xticks(x)
        axes[0].set_xticklabels(labels_tier, fontsize=9)
        axes[0].grid(True, axis='y', alpha=0.3)
        axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v:,.0f}'))
        axes[0].legend(loc='upper left', fontsize=8)
    else:
        axes[0].axis('off')

    # Right: strategy-level
    if strategies:
        x_s = np.arange(len(strategies))
        width = 0.25

        axes[1].bar(x_s - width, incr_s, width, label='Increased Revenue vs Baseline', color='#2ecc71')
        axes[1].bar(x_s, cost_s, width, label='Campaign Cost', color='#e74c3c')
        axes[1].bar(x_s + width, profit_s, width, label='Incremental Profit (Rev − Cost)', color='#3498db')

        axes[1].set_title('Retention vs Acquisition: Profit & ROI', fontsize=12, fontweight='bold')
        axes[1].set_ylabel('Incremental uplift / Cost / Profit per 3 months ($)')
        axes[1].set_xticks(x_s)
        axes[1].set_xticklabels(strategies, fontsize=10)
        axes[1].grid(True, axis='y', alpha=0.3)
        axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v:,.0f}'))
        axes[1].legend(loc='upper left', fontsize=8)
    else:
        axes[1].axis('off')

    plt.tight_layout()
    file_path = f"{output_path}/scenario2_profit_roi.png"
    plt.savefig(file_path, dpi=150, bbox_inches='tight')
    plt.close()
    logger.info(f"   Saved: {file_path}")
    return file_path

def plot_scenario_3_profit_roi(s3_data, output_path):
    """Bar chart: incremental revenue, campaign cost and profit + ROI annotation."""
    incr = s3_data.get('incremental_revenue')
    if incr is None:
        return

    n_target = s3_data.get('n_target')
    cost_per = SUBSCRIPTION_CONFIG.get('campaign_cost_per_user')
    if n_target is None or cost_per is None:
        return

    cost = n_target * cost_per
    profit = incr - cost
    roi = s3_data.get('roi')

    labels = ['Incremental Revenue', 'Campaign Cost', 'Profit']
    values = [incr, cost, profit]
    colors = ['#2ecc71', '#e74c3c', '#3498db']

    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(labels))
    bars = ax.bar(x, values, color=colors, width=0.6)

    ax.set_title('Scenario 3: Subscription – Profit & ROI', fontsize=14, fontweight='bold')
    ax.set_ylabel('Amount per 3 months ($)')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=10)
    ax.grid(True, axis='y', alpha=0.3)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v:,.0f}'))

    max_val = max(values) if values else 0
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                val + (max_val * 0.03 if max_val > 0 else 0.5),
                f'${val:,.0f}', ha='center', va='bottom', fontsize=10)

    if roi is not None:
        profit_bar = bars[-1]
        ax.text(profit_bar.get_x() + profit_bar.get_width() / 2,
                profit_bar.get_height() + (max_val * 0.12 if max_val > 0 else 1.0),
                f'ROI: {roi:.1f}%', ha='center', va='bottom', fontsize=11, color='#f39c12', fontweight='bold')

    plt.tight_layout()
    file_path = f"{output_path}/scenario3_profit_roi.png"
    plt.savefig(file_path, dpi=150, bbox_inches='tight')
    plt.close()
    logger.info(f"   Saved: {file_path}")
    return file_path

In [133]:
def plot_scenario_1(user_df, s1_df, output_path):
    """Grouped bar: baseline vs uplifted revenue per segment."""
    if s1_df.empty:
        return

    fig, ax = plt.subplots(figsize=(10, 5))
    segments = s1_df['Segment'].tolist()
    baseline = s1_df['baseline_revenue'].tolist()
    uplifted = s1_df['uplifted_revenue'].tolist()

    x = np.arange(len(segments))
    width = 0.35
    ax.bar(x - width / 2, baseline, width, label='Baseline', color='#3498db')
    bars2 = ax.bar(x + width / 2, uplifted, width, label='After AOV Uplift', color='#2ecc71')

    # incremental labels on uplifted bars
    for i, (b, u) in enumerate(zip(baseline, uplifted)):
        ax.text(x[i] + width / 2, u + max(uplifted) * 0.02,
                f'+${u - b:,.0f}', ha='center', va='bottom', fontsize=9, color='#27ae60', fontweight='bold')

    ax.set_title('Scenario 1: AOV Uplift', fontsize=14, fontweight='bold')
    ax.set_ylabel('Revenue per 3 months ($)')
    ax.set_xticks(x)
    ax.set_xticklabels(segments)
    ax.legend()
    ax.grid(True, axis='y', alpha=0.3)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v:,.0f}'))
    plt.tight_layout()

    file_path = f"{output_path}/scenario1_aov_uplift.png"
    plt.savefig(file_path, dpi=150, bbox_inches='tight')
    plt.close()
    logger.info(f"   Saved: {file_path}")
    return file_path

def plot_scenario_2(s2_data, output_path):
    """Two panels: per-tier baseline vs retention | retention vs acquisition totals."""
    tier_data = s2_data['tier_data']
    retention_rows = s2_data['retention_rows']
    totals = s2_data['totals']
    acq = s2_data['acquisition']

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # --- left panel: per-tier baseline vs retention ---
    tiers = [t['tier'] for t in tier_data]
    baseline = [t['baseline_revenue'] for t in tier_data]
    ret_revs = []
    has_retention = False
    for r in retention_rows:
        if r['ret_incr'] is not None:
            ret_revs.append(r['baseline_revenue'] + r['ret_incr'])
            has_retention = True
        else:
            ret_revs.append(None)

    x = np.arange(len(tiers))
    width = 0.35
    axes[0].bar(x - width / 2, baseline, width, label='Baseline', color='#3498db')
    if has_retention:
        ret_plot = [v if v is not None else 0 for v in ret_revs]
        axes[0].bar(x + width / 2, ret_plot, width, label='After Freq Uplift', color='#e67e22')
        for i, (b, r) in enumerate(zip(baseline, ret_revs)):
            if r is not None:
                axes[0].text(x[i] + width / 2, r + max(baseline) * 0.02,
                             f'+${r - b:,.0f}', ha='center', va='bottom', fontsize=8, color='#e67e22')

    axes[0].set_title('Per-Tier: Baseline vs Retention', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Revenue per 3 months ($)')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(tiers, fontsize=9)
    axes[0].legend()
    axes[0].grid(True, axis='y', alpha=0.3)
    axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v:,.0f}'))

    # --- right panel: retention vs acquisition incremental ---
    labels = ['Retention', 'Acquisition']
    values = [totals['ret_incr'] or 0, acq['acq_rev'] or 0]
    colors = ['#e67e22', '#27ae60']
    bars = axes[1].bar(labels, values, color=colors, width=0.5)

    for bar, val in zip(bars, values):
        if val > 0:
            axes[1].text(bar.get_x() + bar.get_width() / 2, val + max(values) * 0.02,
                         f'${val:,.0f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

    axes[1].set_title('Retention vs Acquisition: Incremental Revenue', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Incremental Revenue ($)')
    axes[1].grid(True, axis='y', alpha=0.3)
    axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v:,.0f}'))

    if totals['ret_incr'] is None and acq['acq_rev'] is None:
        axes[1].text(0.5, 0.5, 'Fill in parameters\nto populate',
                     ha='center', va='center', transform=axes[1].transAxes, fontsize=13, color='gray')

    plt.tight_layout()
    file_path = f"{output_path}/scenario2_churn_reduction.png"
    plt.savefig(file_path, dpi=150, bbox_inches='tight')
    plt.close()
    logger.info(f"   Saved: {file_path}")
    return file_path

def plot_scenario_3(s3_data, output_path):
    """Bar chart: subscriber freq vs target non-sub freq vs projected converted freq."""
    labels = ['Subscribers', 'Target\nNon-Subs']
    values = [s3_data['freq_sub'], s3_data['freq_target']]
    colors = ['#2ecc71', '#e74c3c']

    if s3_data['freq_projected'] is not None:
        labels.append('Projected\nConverted')
        values.append(s3_data['freq_projected'])
        colors.append('#f39c12')

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(labels, values, color=colors, width=0.45)

    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, val + max(values) * 0.015,
                f'{val:.2f}', ha='center', va='bottom', fontsize=13, fontweight='bold')

    # annotate the gap
    gap = s3_data['freq_gap']
    ax.annotate(f'gap = {gap:.2f}',
                xy=(0, s3_data['freq_target']), xytext=(0.3, (s3_data['freq_sub'] + s3_data['freq_target']) / 2),
                fontsize=9, color='gray',
                arrowprops=dict(arrowstyle='->', color='gray'))

    ax.set_title('Scenario 3: Subscription — Order Frequency', fontsize=14, fontweight='bold')
    ax.set_ylabel('Avg Orders per 3 Months')
    ax.grid(True, axis='y', alpha=0.3)
    ax.set_ylim(0, max(values) * 1.15)
    plt.tight_layout()

    file_path = f"{output_path}/scenario3_subscription.png"
    plt.savefig(file_path, dpi=150, bbox_inches='tight')
    plt.close()
    logger.info(f"   Saved: {file_path}")
    return file_path

In [134]:
# Run value estimation scenarios
user_baseline_df = load_baseline_data_for_value_estimation()

s1_results = scenario_aov_uplift(user_baseline_df, AOV_UPLIFT_CONFIG)
s1b_results = scenario_targeted_aov_uplift(user_baseline_df, TARGETED_AOV_CONFIG)
s2_results = scenario_churn_reduction(user_baseline_df, CHURN_REDUCTION_CONFIG)
s3_results = scenario_subscription_conversion(user_baseline_df, SUBSCRIPTION_CONFIG)

# Save value estimation results
value_output_path = f"{OUTPUT_DIR}/value_estimation"
os.makedirs(value_output_path, exist_ok=True)

if s1_results is not None:
    s1_results.to_csv(f"{value_output_path}/scenario1_aov_uplift.csv", index=False)

# Generate value estimation visualizations
logger.info("\n" + "="*80)
logger.info("GENERATING VALUE ESTIMATION VISUALIZATIONS")
logger.info("="*80)

# AOV distribution plot
plot_aov_distribution(user_baseline_df, output_path=value_output_path)

# Scenario 1: AOV Uplift plots
plot_scenario_1(user_baseline_df, s1_results, value_output_path)
plot_scenario_1_profit_roi(s1_results, value_output_path)

# Scenario 1b: Targeted AOV Uplift plot
plot_targeted_aov_uplift(s1b_results, value_output_path)

# Scenario 2: Churn Reduction plots
plot_scenario_2(s2_results, value_output_path)
plot_scenario_2_profit_roi(s2_results, value_output_path)

# Scenario 3: Subscription plots
plot_scenario_3(s3_results, value_output_path)
plot_scenario_3_profit_roi(s3_results, value_output_path)

logger.info(f"\n Saved value estimation results to: {value_output_path}")

2026-02-12 23:09:24,383 - INFO - Loaded 8,441 users for value estimation
2026-02-12 23:09:24,383 - INFO - 
2026-02-12 23:09:24,384 - INFO -  SCENARIO 1: AOV UPLIFT
2026-02-12 23:09:24,384 - INFO - ======================================================================
2026-02-12 23:09:24,386 - INFO - 
Segment                 n    Freq      AOV     Baseline  Uplift    Incr. Rev      ROI
2026-02-12 23:09:24,386 - INFO - ------------------------------------------------------------------------------------------
2026-02-12 23:09:24,386 - INFO - Golden Whales         374   2.661 $  81.15 $ 80,769.82     8% $  6,057.74  439.9%
2026-02-12 23:09:24,386 - INFO - High Potential      4,637   0.759 $  23.35 $ 82,147.81    15% $ 12,322.17  165.7%
2026-02-12 23:09:24,387 - INFO - Casual Walk-in      3,430   0.277 $  28.02 $ 26,637.08    10% $  2,663.71   55.3%
2026-02-12 23:09:24,387 - INFO - 
2026-02-12 23:09:24,387 - INFO - SCENARIO 1b: TARGETED AOV THRESHOLD UPLIFT
2026-02-12 23:09:24,387 - INFO - 

## Pipeline Complete 

A summary of results:

### Models & Analysis
-  **Feature engineering** - 30+ temporal features per order
-  **Model training** - 4 algorithms trained and compared
-  **Best model selected** - Based on PR-AUC metric
-  **SHAP analysis** - Both order-level and user-level interpretability
-  **Segment analysis** - Performance breakdown by customer segment

### Business Insights
-  **Revenue impact** - Expected revenue at risk by segment
-  **Value estimation** - ROI calculations for 3 strategic scenarios
-  **Prioritization** - Which customers to target first

### Outputs Generated
- **Features** - `outputs/repurchase/repurchase_features.csv`, train/test splits
- **Models** - `outputs/repurchase/models/will_repurchase_best_model.pkl` + SHAP values
- **Visualizations** - `outputs/repurchase/visualizations/*.png` (15+ plots)
- **Revenue analysis** - `outputs/repurchase/revenue_impact/*.csv` + plots
- **Value estimation** - `outputs/repurchase/value_estimation/*.csv` + scenario plots

### Next Steps
1. **Review SHAP plots** - Understand what drives churn in each segment
2. **Analyze revenue impact** - Identify high-value at-risk customers
3. **Select intervention strategy** - Choose from AOV uplift, retention, or subscription scenarios
4. **Deploy model** - Integrate predictions into CRM/marketing automation
5. **Monitor performance** - Track retention rates and campaign ROI over time

In [135]:
logger.info("\n" + "="*80)
logger.info("PIPELINE COMPLETE")
logger.info("="*80)
logger.info(f"Best Model: {best_model_name}")
logger.info(f"ROC-AUC: {best_metrics['roc_auc']:.4f}")
logger.info(f"PR-AUC: {best_metrics['pr_auc']:.4f}")
logger.info(f"F1-Score: {best_metrics['f1_score']:.4f}")
logger.info(f"\nOutputs saved to:")
logger.info(f"  - {OUTPUT_DIR}/  (features)")
logger.info(f"  - {MODELS_PATH}/  (models)")
logger.info(f"  - {revenue_output_path}/  (revenue analysis)")
logger.info(f"  - {value_output_path}/  (value estimation)")

2026-02-12 23:09:24,928 - INFO - 
2026-02-12 23:09:24,929 - INFO - PIPELINE COMPLETE
2026-02-12 23:09:24,929 - INFO - ================================================================================
2026-02-12 23:09:24,929 - INFO - Best Model: XGBoost
2026-02-12 23:09:24,930 - INFO - ROC-AUC: 0.8155
2026-02-12 23:09:24,930 - INFO - PR-AUC: 0.8607
2026-02-12 23:09:24,930 - INFO - F1-Score: 0.7960
2026-02-12 23:09:24,930 - INFO - 
Outputs saved to:
2026-02-12 23:09:24,930 - INFO -   - outputs/repurchase/  (features)
2026-02-12 23:09:24,930 - INFO -   - outputs/repurchase/models/  (models)
2026-02-12 23:09:24,931 - INFO -   - outputs/repurchase/revenue_impact/  (revenue analysis)
2026-02-12 23:09:24,931 - INFO -   - outputs/repurchase/value_estimation/  (value estimation)
